# tiny-pairvc: training notebook (Kaggle & Colab)

Train **PairVC**, a tiny real-time voice converter, on a **free Kaggle or Colab
GPU**. PairVC learns from **paired audio**: each source clip comes with the same
speech already converted to a target voice by a big (slow) teacher model, and
PairVC learns to reproduce it while staying small and causal enough to run
live, 20 ms at a time.

This notebook is **self-contained**: all the code (config, data, model, losses,
discriminators, training loop) is copied in from the `tiny-pairvc` repo, with
the **same default configuration**. The checkpoints it writes work with the
repo's `pairvc-convert`, `pairvc-export` and `pairvc-train --resume`.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Your data | *Add Input → Datasets → your paired dataset* | Google Drive, or upload a zip |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**No data yet?** Set `USE_DIGITS_DATA = True`: the notebook builds a small
*real speech* paired dataset (spoken digits, with a pitch-shift "teacher": the
repo's README results). Or `USE_TOY_DATA = True` / `SMOKE_TEST = True`: a tiny
synthetic dataset so every cell can be tried quickly.

**Sections:** 1. Install · 2. Config · 3. Data · 4. Model · 5. Losses &
discriminators · 6. Training code · 7. Live dashboard · 8. Train · 9. Results
(curves, listen, streaming speed, objective metrics: MCD, F0, DNSMOS) · 10. Download the trained model

## 1. Install

Kaggle and Colab already have PyTorch, NumPy, SciPy and matplotlib; this makes
sure `soundfile` (reads/writes .wav) is there too.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile>=0.12"])

import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Config

### 2.1 Config classes (same as `src/pairvc/config.py`)

All hyperparameters live here, grouped by what they control.

Configs are frozen dataclasses: they cannot be changed by accident after
creation, and they are saved inside every checkpoint so a run can be reproduced
and a trained model can be rebuilt without remembering any flag.

In [ ]:
from dataclasses import asdict, dataclass
from math import prod
from pathlib import Path
from typing import Any

DISCRIMINATOR_KINDS = ("time", "period", "resolution", "band")


@dataclass(frozen=True, slots=True)
class DataConfig:
    """Where the paired clips are and how they are cut into training examples.

    Layout expected under `data_dir` (see docs/data.md):
        <data_dir>/<split>/<id>_original.wav      source speech (any speaker)
        <data_dir>/<split>/<id>_converted_<k>.wav  same speech in target voice k
    """

    data_dir: Path = Path("data")
    train_split: str = "train"
    val_split: str = "val"
    sample_rate: int = 16_000
    # Length of one training example: 16_000 = 1 s at 16 kHz. Must be a multiple of
    # the model hop (320). Training memory grows with batch_size x segment length:
    # ~0.8 GB of activations per second of audio in the batch (see docs/training.md).
    # The generator is fully convolutional, so it converts clips of any length later.
    segment_samples: int = 16_000

    def __post_init__(self) -> None:
        if self.sample_rate <= 0:
            raise ValueError("sample_rate must be positive")
        if self.segment_samples <= 0:
            raise ValueError("segment_samples must be positive")


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Shape of the generator (the voice converter itself)."""

    n_speakers: int = 2  # number of target voices
    channels: int = 32  # width of the first layer; doubles after every stride
    latent_dim: int = 64  # size of one latent frame between encoder and decoder
    speaker_dim: int = 64  # size of the target-speaker embedding
    strides: tuple[int, ...] = (2, 4, 5, 8)  # product = hop = samples per latent frame
    residual_units: int = 3  # residual units per block (dilations 1, 3, 9)
    kernel_size: int = 7
    # The output is compared with the target `lookahead_frames` frames *earlier*:
    # the model may hear a little future before it speaks (2 x 20 ms = 40 ms).
    # This is extra latency in streaming, traded for quality.
    lookahead_frames: int = 2

    def __post_init__(self) -> None:
        if self.n_speakers <= 0:
            raise ValueError("n_speakers must be positive")
        if not self.strides or any(s <= 0 for s in self.strides):
            raise ValueError("strides must be a non-empty tuple of positive ints")
        if self.lookahead_frames < 0:
            raise ValueError("lookahead_frames must be >= 0")

    @property
    def hop_length(self) -> int:
        """Audio samples per latent frame (320 = 20 ms at 16 kHz by default)."""
        return prod(self.strides)

    @property
    def lookahead_samples(self) -> int:
        return self.lookahead_frames * self.hop_length


@dataclass(frozen=True, slots=True)
class DiscConfig:
    """Which discriminators judge the generated audio, and their sizes.

    "time":       multi-scale waveform discriminator (HiFi-GAN MSD)
    "period":     multi-period discriminator (HiFi-GAN MPD)
    "resolution": multi-resolution STFT magnitude discriminator (UnivNet MRD)
    "band":       multi-band complex-STFT discriminator (DAC MBD)
    """

    kinds: tuple[str, ...] = ("time",)
    lrelu_slope: float = 0.1
    # time
    time_scales: int = 3
    # period
    periods: tuple[int, ...] = (2, 3, 5, 7, 11)
    period_kernel_size: int = 5
    period_stride: int = 3
    # resolution: (n_fft, hop_length, win_length)
    resolutions: tuple[tuple[int, int, int], ...] = (
        (1024, 90, 450),  # ~28 ms window, 5.6 ms hop
        (2048, 180, 900),  # ~56 ms window, 11 ms hop
        (512, 36, 180),  # ~11 ms window, 2.3 ms hop
    )
    resolution_channels: int = 32
    # band
    band_fft_sizes: tuple[int, ...] = (2048, 1024, 512)
    band_channels: int = 32
    band_edges: tuple[float, ...] = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)  # fractions of Nyquist

    def __post_init__(self) -> None:
        unknown = set(self.kinds) - set(DISCRIMINATOR_KINDS)
        if not self.kinds or unknown:
            raise ValueError(f"kinds must be a non-empty subset of {DISCRIMINATOR_KINDS}")


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """Optimisation and bookkeeping settings for one training run."""

    output_dir: Path = Path("checkpoints/run")
    seed: int = 42
    max_steps: int = 500_000  # GAN vocoders need many steps; stop earlier any time
    batch_size: int = 16  # per process (per GPU)
    g_lr: float = 1e-4  # generator learning rate
    d_lr: float = 1e-4  # discriminator learning rate
    betas: tuple[float, float] = (0.8, 0.99)  # HiFi-GAN values
    weight_decay: float = 0.0
    lr_decay: float = 0.9999996  # multiply both LRs by this after every step
    grad_clip: float = 0.0  # 0 = no clipping
    lambda_adv: float = 1.0  # adversarial loss weight
    lambda_fm: float = 2.0  # feature-matching loss weight
    lambda_mel: float = 15.0  # multi-scale mel loss weight
    # Mel loss only before this step: the discriminators are skipped (~3x faster
    # steps on CPU) while the generator learns the coarse sound; 0 = GAN from the start.
    gan_start_step: int = 0
    precision: str = "auto"  # "auto" | "fp32" | "bf16"
    grad_checkpointing: bool = False  # less GPU memory, ~30% slower
    num_workers: int = 4
    train_limit: int | None = None  # use only N training pairs (tiny overfit runs)
    val_limit: int | None = None
    log_every: int = 50  # steps between log lines
    eval_every: int = 1000  # steps between validations (+ checkpoint + audio sample)
    device: str = "auto"  # "auto" | "cpu" | "cuda" | "mps"
    resume: Path | None = None  # checkpoint to continue from

    def __post_init__(self) -> None:
        if self.batch_size <= 0:
            raise ValueError("batch_size must be positive")
        if self.max_steps <= 0:
            raise ValueError("max_steps must be positive")
        if self.precision not in {"auto", "fp32", "bf16"}:
            # fp16 is left out on purpose: GAN losses overflow it easily.
            raise ValueError("precision must be auto, fp32 or bf16")
        if not 0.0 < self.lr_decay <= 1.0:
            raise ValueError("lr_decay must be in (0, 1]")


def config_to_dict(config: Any) -> dict[str, Any]:
    """Turn a config into plain types so it can be stored in a checkpoint."""
    return {k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()}

### 2.2 Settings for this run

**Where is your data?** A folder with `train/` and `val/` inside, each holding
`<id>_original.wav` + `<id>_converted_<k>.wav` files (see the repo's
`docs/data.md`). On Kaggle, upload it as a Dataset and add it as input: the
cell below finds it under `/kaggle/input` by itself. On Colab, put it on Google
Drive (`USE_GOOGLE_DRIVE = True`) and set `DATA_DIR`, or upload a zip and unzip
it to `/content/data`.

Only two things adapt to the GPU, without changing what is learned:

- **batch size**: from the GPU memory (training needs ~0.8 GB of activations
  per second of audio in the batch; gradient checkpointing cuts the generator's
  share by ~10x for ~30% more time);
- **precision**: bf16 on GPUs with native support (A100, H100, L4, RTX 30xx/40xx),
  fp32 on older ones (T4, P100). fp16 is not used: GAN losses overflow it.

In [ ]:
import dataclasses
import os
from pathlib import Path

SMOKE_TEST = False          # True = tiny quick check of the whole notebook (toy data, tiny model)
USE_TOY_DATA = False        # True = synthetic paired data instead of yours
USE_DIGITS_DATA = False     # True = real spoken digits + a pitch-shift teacher (downloads 16 MB)
USE_GOOGLE_DRIVE = False    # Colab only: data + checkpoints on Drive (survive disconnects)
DATA_DIR = None             # None = find it automatically (Kaggle input, /content/data, ./data)
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists
TIME_BUDGET_HOURS = 11.0    # stop cleanly (after saving) before the 12 h session limit
N_SPEAKERS = None           # None = number of target voices found in the data

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ  # set by every Colab runtime
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/tiny-pairvc/run")
elif ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working/checkpoints/run")
elif ON_COLAB:
    OUTPUT_DIR = Path("/content/checkpoints/run")
else:
    OUTPUT_DIR = Path("checkpoints/notebook-run")


def looks_like_paired_data(path: Path) -> bool:
    return (path / "train").is_dir() and any((path / "train").glob("*_original.wav"))


def find_data_dir() -> Path | None:
    """First folder containing train/<id>_original.wav, in the usual places."""
    roots = [Path("/kaggle/input"), Path("/content/data"), Path("/content/drive/MyDrive"), Path("data")]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root, *sorted(p for p in root.rglob("*") if p.is_dir() and p.name != "train")]
        for path in candidates[:500]:
            if looks_like_paired_data(path):
                return path
    return None


if SMOKE_TEST or USE_TOY_DATA or USE_DIGITS_DATA:
    DATA_DIR = None  # written in section 3, once write_toy_dataset / write_digits_dataset are defined
elif DATA_DIR is None:
    DATA_DIR = find_data_dir()
    if DATA_DIR is None:
        raise FileNotFoundError(
            "No paired data found. Add your dataset as input (Kaggle) / set DATA_DIR, "
            "or set USE_TOY_DATA = True to try the notebook on synthetic data."
        )
print("data:", DATA_DIR or ("spoken digits" if USE_DIGITS_DATA and not SMOKE_TEST else "toy data (synthetic)"))

# --- device, precision, batch size ------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_config = DataConfig(data_dir=Path(DATA_DIR or "toy_data"))
model_config = ModelConfig()
disc_config = DiscConfig()

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_gb = props.total_memory / 1e9
    precision = "bf16" if props.major >= 8 else "fp32"  # native bf16 needs compute capability 8.x+
    # ~0.8 GB per second of audio in the batch; checkpointing below 40 GB. Lower it if "out of memory".
    batch_size = 32 if gpu_gb >= 70 else 16 if gpu_gb >= 14 else 8
    grad_checkpointing = gpu_gb < 40
    print(f"GPU: {props.name} ({gpu_gb:.0f} GB), compute capability {props.major}.{props.minor}")
else:
    precision, batch_size, grad_checkpointing = "fp32", 4, False
    print("No GPU: enable one in the notebook settings (CPU is fine only for SMOKE_TEST).")

train_config = TrainConfig(
    output_dir=OUTPUT_DIR,
    batch_size=batch_size,
    precision=precision,
    grad_checkpointing=grad_checkpointing,
    num_workers=min(4, os.cpu_count() or 1),
    device=device.type,
    log_every=50,
    eval_every=1000,
)
if USE_DIGITS_DATA and not SMOKE_TEST:
    data_config = dataclasses.replace(data_config, segment_samples=8000)  # a digit lasts ~0.5 s

if SMOKE_TEST:
    OUTPUT_DIR = OUTPUT_DIR.parent / "smoke-test"  # never mixed with a real run
    data_config = dataclasses.replace(data_config, segment_samples=3200)
    model_config = ModelConfig(channels=8, latent_dim=16, speaker_dim=16, residual_units=2)
    disc_config = DiscConfig(kinds=("resolution",))
    train_config = dataclasses.replace(
        train_config, output_dir=OUTPUT_DIR, max_steps=40, batch_size=4, log_every=2,
        eval_every=20, num_workers=0, g_lr=1e-3,
    )
    TIME_BUDGET_HOURS = 0.5
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, cfg in [("DataConfig", data_config), ("ModelConfig", model_config),
                  ("DiscConfig", disc_config), ("TrainConfig", train_config)]:
    print(f"\n{name}:")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key:20} {value}")

## 3. Data

### 3.1 Dataset code (same as `src/pairvc/data.py`)

Paired training data: source speech + the same speech in a target voice.

PairVC learns from *pairs*. A large, slow teacher voice-conversion model was
run offline on every source clip, once per target voice; PairVC is trained to
reproduce the teacher's output, sample for sample, while being small and causal
enough for real time (teacher -> student distillation on synthetic parallel
data). See docs/data.md for how to build such a dataset.

Files in one split directory:

    <id>_original.wav       source speech, any speaker
    <id>_converted_0.wav    the same words, converted to target voice 0
    <id>_converted_1.wav    ... target voice 1, and so on

Every (source, converted_k) couple is one training example with label k, so a
source with 2 conversions gives 2 examples.

In [ ]:
import math
import random
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly
from torch import Tensor
from torch.utils.data import Dataset

SOURCE_SUFFIX = "_original.wav"
TARGET_PATTERN = re.compile(r"^(?P<id>.+)_converted_(?P<speaker>\d+)\.wav$")


@dataclass(frozen=True, slots=True)
class Pair:
    source: Path
    target: Path
    speaker: int


def find_pairs(split_dir: Path) -> list[Pair]:
    """Every (source, converted_k) couple in a directory, sorted (same order on every machine)."""
    split_dir = Path(split_dir)
    if not split_dir.is_dir():
        raise FileNotFoundError(f"split directory not found: {split_dir}")
    pairs = []
    for target in sorted(split_dir.glob("*_converted_*.wav")):
        match = TARGET_PATTERN.match(target.name)
        if match is None:
            continue
        source = split_dir / f"{match['id']}{SOURCE_SUFFIX}"
        if source.exists():
            pairs.append(Pair(source, target, int(match["speaker"])))
    return pairs


def load_audio(path: Path, sample_rate: int) -> Tensor:
    """Read a sound file as mono float32 in [-1, 1] at `sample_rate`: [T]."""
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)  # any bit depth -> float
    audio = audio.mean(axis=1)  # stereo -> mono
    if sr != sample_rate:
        g = math.gcd(sr, sample_rate)
        audio = resample_poly(audio, sample_rate // g, sr // g).astype(np.float32)
    return torch.from_numpy(np.ascontiguousarray(audio))


def crop_pair(source: Tensor, target: Tensor, length: int, random_crop: bool) -> tuple[Tensor, Tensor]:
    """Cut the same window out of both clips, zero-padding clips that are too short."""
    n = min(source.shape[-1], target.shape[-1])  # the teacher may add or drop a few samples
    source, target = source[:n], target[:n]
    if n < length:
        return torch.nn.functional.pad(source, (0, length - n)), torch.nn.functional.pad(
            target, (0, length - n)
        )
    start = random.randint(0, n - length) if random_crop else 0
    return source[start : start + length], target[start : start + length]


class PairedVoiceDataset(Dataset):
    """Returns {"source": [T], "target": [T], "speaker": long} with T = segment_samples.

    Training takes a random window of each clip (new each epoch); validation
    always takes the start of the clip, so its numbers are comparable.
    """

    def __init__(
        self,
        split_dir: Path,
        sample_rate: int,
        segment_samples: int,
        train: bool = False,
        limit: int | None = None,
    ) -> None:
        self.pairs = find_pairs(split_dir)
        if limit is not None:
            self.pairs = self.pairs[:limit]
        if not self.pairs:
            raise ValueError(f"no '<id>_original.wav' + '<id>_converted_<k>.wav' pairs in {split_dir}")
        self.sample_rate = sample_rate
        self.segment_samples = segment_samples
        self.train = train

    @property
    def speakers(self) -> list[int]:
        return sorted({p.speaker for p in self.pairs})

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        pair = self.pairs[index]
        source = load_audio(pair.source, self.sample_rate)
        target = load_audio(pair.target, self.sample_rate)
        source, target = crop_pair(source, target, self.segment_samples, random_crop=self.train)
        return {"source": source, "target": target, "speaker": torch.tensor(pair.speaker)}


def _toy_voice(
    f0: np.ndarray, envelope: np.ndarray, sample_rate: int, pitch_scale: float, brightness: float
) -> np.ndarray:
    """7 harmonics of a pitch contour; brightness sets how fast they fade (the timbre)."""
    phase = 2 * np.pi * np.cumsum(f0 * pitch_scale) / sample_rate
    harmonics = sum(np.sin(h * phase) * brightness**h for h in range(1, 8))
    return (0.15 * envelope * harmonics).astype(np.float32)


def write_toy_dataset(
    root: Path,
    n_clips: int = 16,
    seconds: float = 1.0,
    sample_rate: int = 16_000,
    n_speakers: int = 2,
    seed: int = 0,
) -> Path:
    """Write a tiny synthetic paired dataset: no real data needed.

    Each "source" is a buzzy vowel-like tone with a random pitch contour; voice
    k is the same contour with pitch scaled by (k + 2) / 2 and a different
    timbre. Useful to check the whole pipeline (tests, notebook smoke test);
    a model trained on it will not convert real voices.
    """
    rng = np.random.default_rng(seed)
    n = int(seconds * sample_rate)
    t = np.arange(n) / sample_rate
    for split, n_split in (("train", n_clips), ("val", max(2, n_clips // 4))):
        split_dir = Path(root) / split
        split_dir.mkdir(parents=True, exist_ok=True)
        for i in range(n_split):
            f0 = rng.uniform(100, 220) * (1 + 0.1 * np.sin(2 * np.pi * rng.uniform(0.5, 3) * t))
            envelope = np.clip(np.sin(np.pi * t / seconds) * 1.5, 0, 1)
            source = _toy_voice(f0, envelope, sample_rate, pitch_scale=1.0, brightness=0.6)
            sf.write(split_dir / f"clip{i:04d}{SOURCE_SUFFIX}", source, sample_rate)
            for k in range(n_speakers):
                target = _toy_voice(f0, envelope, sample_rate, (k + 2) / 2, 0.4 + 0.2 * k)
                sf.write(split_dir / f"clip{i:04d}_converted_{k}.wav", target, sample_rate)
    return Path(root)

### 3.2 Real speech without a teacher model (same as `src/pairvc/digits.py`)

A small *real speech* paired dataset, built in a minute: spoken digits + a DSP teacher.

The real recipe needs a big teacher VC model (docs/data.md). To show that PairVC
learns on real speech without one, this module plays the teacher with plain
signal processing:

- source speech: the Free Spoken Digit Dataset (FSDD, CC BY-SA 4.0), 3000 clips
  of 6 people saying "zero" ... "nine", 8 kHz, ~0.5 s each;
- target voice k: the same clip pitch-shifted by `SHIFTS[k]` semitones with a
  phase vocoder. Pitch *and* formants move together (a higher / deeper voice),
  and the timing is kept, which is what PairVC's loss needs.

Validation uses a speaker that is never seen in training ("theo"), so the
numbers say how well PairVC converts a voice it never heard.

    from pairvc.digits import write_digits_dataset
    write_digits_dataset("digits_data")      # downloads FSDD once (16 MB)

In [ ]:
import io
import math
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly


FSDD_URL = "https://codeload.github.com/Jakobovski/free-spoken-digit-dataset/zip/refs/heads/master"
SHIFTS = (5.0, -5.0)  # semitones: target voice 0 is higher, voice 1 is deeper
VAL_SPEAKER = "theo"  # held out: validation is on an unseen source voice


def phase_vocoder_stretch(audio: np.ndarray, rate: float, n_fft: int = 512, hop: int = 128) -> np.ndarray:
    """Play `audio` `rate` times faster without changing its pitch (rate < 1 = longer).

    Classic phase vocoder: read the STFT at fractional frame positions, keep each
    bin's magnitude and advance its phase by the measured instantaneous frequency.
    """
    window = np.hanning(n_fft)
    pad = np.pad(audio, (n_fft // 2, n_fft // 2 + n_fft))
    n_frames = 1 + (len(pad) - n_fft) // hop
    frames = np.stack([pad[i * hop : i * hop + n_fft] * window for i in range(n_frames)])
    spec = np.fft.rfft(frames, axis=1)  # [frames, bins]

    expected = 2 * np.pi * hop * np.arange(n_fft // 2 + 1) / n_fft  # phase advance per hop
    positions = np.arange(0, n_frames - 1, rate)
    phase = np.angle(spec[0])
    out_frames = []
    for pos in positions:
        i, frac = int(pos), pos - int(pos)
        magnitude = (1 - frac) * np.abs(spec[i]) + frac * np.abs(spec[i + 1])
        out_frames.append(magnitude * np.exp(1j * phase))
        delta = np.angle(spec[i + 1]) - np.angle(spec[i]) - expected
        delta -= 2 * np.pi * np.round(delta / (2 * np.pi))  # wrap to [-pi, pi]
        phase = phase + expected + delta

    # Overlap-add with the same window, normalised by the summed squared window.
    frames_out = np.fft.irfft(np.stack(out_frames), n=n_fft, axis=1) * window
    length = n_fft + hop * (len(frames_out) - 1)
    out, norm = np.zeros(length), np.zeros(length)
    for j, frame in enumerate(frames_out):
        out[j * hop : j * hop + n_fft] += frame
        norm[j * hop : j * hop + n_fft] += window**2
    out = out / np.maximum(norm, 1e-3)
    return out[n_fft // 2 : n_fft // 2 + round(len(audio) / rate)]


def pitch_shift(audio: np.ndarray, semitones: float) -> np.ndarray:
    """Same duration, pitch (and formants) moved by `semitones`: the toy teacher."""
    ratio = 2 ** (semitones / 12)
    stretched = phase_vocoder_stretch(audio, 1 / ratio)  # ratio x longer, same pitch
    up, down = (round(1000 / ratio), 1000)
    g = math.gcd(up, down)
    shifted = resample_poly(stretched, up // g, down // g)  # back to the original length, pitch x ratio
    shifted = np.pad(shifted, (0, max(0, len(audio) - len(shifted))))[: len(audio)]
    return shifted.astype(np.float32)


def download_fsdd(cache_dir: Path) -> Path:
    """Download and unpack FSDD once; returns the folder holding the 3000 wav files."""
    recordings = Path(cache_dir) / "fsdd" / "recordings"
    if recordings.is_dir() and any(recordings.glob("*.wav")):
        return recordings
    recordings.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(FSDD_URL, timeout=120) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    for name in archive.namelist():
        if "/recordings/" in name and name.endswith(".wav"):
            (recordings / Path(name).name).write_bytes(archive.read(name))
    return recordings


def write_digits_dataset(
    root: Path,
    sample_rate: int = 16_000,
    clips_per_speaker: int = 100,
    val_clips: int = 40,
    cache_dir: Path | None = None,
    shifts: tuple[float, ...] = SHIFTS,
    seed: int = 0,
) -> Path:
    """Write <root>/{train,val}/<id>_original.wav + <id>_converted_<k>.wav from FSDD.

    `clips_per_speaker` clips of each of the 5 training speakers go to train/,
    `val_clips` clips of the held-out speaker go to val/. Audio is resampled from
    8 kHz to `sample_rate`, peak-normalised, and padded with 0.1 s of silence.
    """
    recordings = download_fsdd(Path(cache_dir) if cache_dir else Path(root) / ".cache")
    by_speaker: dict[str, list[Path]] = {}
    for path in sorted(recordings.glob("*.wav")):
        by_speaker.setdefault(path.stem.split("_")[1], []).append(path)  # <digit>_<speaker>_<n>.wav

    rng = np.random.default_rng(seed)
    pad = np.zeros(sample_rate // 10, dtype=np.float32)
    for speaker, paths in sorted(by_speaker.items()):
        split, n = ("val", val_clips) if speaker == VAL_SPEAKER else ("train", clips_per_speaker)
        split_dir = Path(root) / split
        split_dir.mkdir(parents=True, exist_ok=True)
        for path in rng.permutation(paths)[:n]:
            audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
            audio = audio.mean(axis=1)
            g = math.gcd(sr, sample_rate)
            audio = resample_poly(audio, sample_rate // g, sr // g).astype(np.float32)
            audio = np.concatenate([pad, 0.9 * audio / max(np.abs(audio).max(), 1e-6), pad])
            clip_id = Path(path).stem
            sf.write(split_dir / f"{clip_id}{SOURCE_SUFFIX}", audio, sample_rate)
            for k, semitones in enumerate(shifts):
                target = np.clip(pitch_shift(audio, semitones), -1, 1)
                sf.write(split_dir / f"{clip_id}_converted_{k}.wav", target, sample_rate)
    return Path(root)

In [ ]:
if DATA_DIR is None and USE_DIGITS_DATA and not SMOKE_TEST:
    DATA_DIR = write_digits_dataset(Path("digits_data"))
    data_config = dataclasses.replace(data_config, data_dir=DATA_DIR)
elif DATA_DIR is None:  # toy data requested
    DATA_DIR = write_toy_dataset(Path("toy_data"), n_clips=64 if not SMOKE_TEST else 8, seconds=1.0)
    data_config = dataclasses.replace(data_config, data_dir=DATA_DIR)

root = Path(data_config.data_dir)
train_dataset = PairedVoiceDataset(
    root / data_config.train_split, data_config.sample_rate, data_config.segment_samples,
    train=True, limit=train_config.train_limit,
)
val_dataset = PairedVoiceDataset(
    root / data_config.val_split, data_config.sample_rate, data_config.segment_samples,
    limit=train_config.val_limit,
)
speakers = train_dataset.speakers
model_config = dataclasses.replace(model_config, n_speakers=N_SPEAKERS or max(speakers) + 1)
print(f"train: {len(train_dataset):,} pairs | validation: {len(val_dataset):,} pairs | "
      f"target voices: {speakers} -> n_speakers = {model_config.n_speakers}")

### 3.3 Look at (and listen to) one pair
The source on top, the teacher's conversion below: the model learns to turn
the first into the second.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display


def show_spectrogram(audio: torch.Tensor, ax, title: str, sample_rate: int) -> None:
    spec = torch.stft(audio, 512, 128, window=torch.hann_window(512), return_complex=True).abs()
    ax.imshow((spec + 1e-5).log10().numpy(), origin="lower", aspect="auto",
              extent=[0, len(audio) / sample_rate, 0, sample_rate / 2000])
    ax.set_title(title)
    ax.set_ylabel("kHz")


pair = val_dataset.pairs[0]
source, target = load_audio(pair.source, data_config.sample_rate), load_audio(pair.target, data_config.sample_rate)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
show_spectrogram(source, axes[0], f"source: {pair.source.name}", data_config.sample_rate)
show_spectrogram(target, axes[1], f"teacher, voice {pair.speaker}: {pair.target.name}", data_config.sample_rate)
axes[1].set_xlabel("seconds")
plt.tight_layout()
plt.show()
display(Audio(source.numpy(), rate=data_config.sample_rate), Audio(target.numpy(), rate=data_config.sample_rate))

## 4. Model

### 4.1 PairVC code (same as `src/pairvc/model.py`)

PairVC: a small, fully causal waveform-to-waveform voice converter.

    source audio [B, T] --encoder--> latent frames [B, T/320, 64]
                       --decoder (+ target speaker)--> converted audio [B, T]

Every convolution is *causal*: output sample t only depends on input samples
<= t. That is what lets the same network run in real time, 20 ms at a time.

Each layer has two ways to run:

- `forward(x)`            the whole clip at once (training, offline conversion)
- `step(x, states, new)`  one chunk, carrying a small state between chunks
                          (streaming, ONNX export)

Both give the same numbers (tests/test_model.py checks it), so a model trained
with `forward` can be streamed without any conversion.

Streaming state convention: `init_state(batch, out)` appends this layer's
zero states to the list `out`; `step(x, states, new)` reads its states from the
iterator `states` in the same order and appends the updated ones to `new`.

In [ ]:
import math
from collections.abc import Iterator

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.checkpoint import checkpoint


States = Iterator[Tensor]  # states being read by step()
NewStates = list[Tensor]  # states being written by step()


# ---------------------------------------------------------------------------
# Causal building blocks
# ---------------------------------------------------------------------------


class CausalConv1d(nn.Conv1d):
    """Conv1d that only looks at the past (left zero padding).

    With kernel k, dilation d and stride s, one output looks at
    `receptive = d * (k - 1) + 1` input samples. Padding the left by
    `receptive - s` makes an input of length T give exactly T / s outputs.
    """

    def __init__(
        self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1,
        dilation: int = 1,
    ) -> None:  # fmt: skip
        super().__init__(in_channels, out_channels, kernel_size, stride=stride, dilation=dilation)
        self.receptive = dilation * (kernel_size - 1) + 1
        self.left_pad = self.receptive - stride
        if self.left_pad < 0:
            raise ValueError("kernel too small for its stride: samples would be skipped")

    def forward(self, x: Tensor) -> Tensor:
        return super().forward(F.pad(x, (self.left_pad, 0)))

    def init_state(self, batch_size: int, out: NewStates) -> None:
        # The last `left_pad` input samples not yet fully used; zeros = the left padding.
        out.append(torch.zeros(batch_size, self.in_channels, self.left_pad))

    def step(self, x: Tensor, states: States, new: NewStates) -> Tensor:
        x = torch.cat([next(states), x], dim=-1)
        stride = self.stride[0]
        n_out = (x.shape[-1] - self.receptive) // stride + 1
        used = n_out * stride  # where the next output window starts
        new.append(x[..., used:])
        return super().forward(x[..., : used - stride + self.receptive])


class CausalConvTranspose1d(nn.ConvTranspose1d):
    """Upsampling ConvTranspose1d trimmed so that output sample t never uses future input.

    Kernel 2s / stride s: T frames in, exactly T * s samples out.
    """

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int) -> None:
        super().__init__(in_channels, out_channels, kernel_size, stride=stride)
        self.trim = kernel_size - stride  # samples at the end that still wait for the next frame
        if self.trim < 0:
            raise ValueError("kernel_size must be >= stride")
        self.context = math.ceil(self.trim / stride)  # past frames a chunk needs to see

    def forward(self, x: Tensor) -> Tensor:
        y = super().forward(x)
        return y[..., : y.shape[-1] - self.trim]

    def init_state(self, batch_size: int, out: NewStates) -> None:
        out.append(torch.zeros(batch_size, self.in_channels, self.context))

    def step(self, x: Tensor, states: States, new: NewStates) -> Tensor:
        n_new = x.shape[-1]
        x = torch.cat([next(states), x], dim=-1)
        new.append(x[..., x.shape[-1] - self.context :])
        y = super().forward(x)
        start = self.context * self.stride[0]  # drop what the previous chunk already output
        return y[..., start : start + n_new * self.stride[0]]


class ResidualUnit(nn.Module):
    """x + ELU(conv1x1(ELU(dilated causal conv(x))))."""

    def __init__(self, channels: int, dilation: int, kernel_size: int = 7) -> None:
        super().__init__()
        self.conv1 = CausalConv1d(channels, channels, kernel_size, dilation=dilation)
        self.conv2 = CausalConv1d(channels, channels, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        return x + F.elu(self.conv2(F.elu(self.conv1(x))))

    def init_state(self, batch_size: int, out: NewStates) -> None:
        self.conv1.init_state(batch_size, out)
        self.conv2.init_state(batch_size, out)

    def step(self, x: Tensor, states: States, new: NewStates) -> Tensor:
        y = F.elu(self.conv1.step(x, states, new))
        return x + F.elu(self.conv2.step(y, states, new))


class EncoderBlock(nn.Module):
    """Residual units at the current rate, then a strided conv (downsample by `stride`)."""

    def __init__(
        self, in_channels: int, out_channels: int, stride: int, residual_units: int,
        kernel_size: int,
    ) -> None:  # fmt: skip
        super().__init__()
        self.residuals = nn.ModuleList(
            ResidualUnit(in_channels, dilation=3**i, kernel_size=kernel_size) for i in range(residual_units)
        )
        self.down = CausalConv1d(in_channels, out_channels, kernel_size=2 * stride, stride=stride)

    def forward(self, x: Tensor) -> Tensor:
        for unit in self.residuals:
            x = unit(x)
        return F.elu(self.down(x))

    def init_state(self, batch_size: int, out: NewStates) -> None:
        for unit in self.residuals:
            unit.init_state(batch_size, out)
        self.down.init_state(batch_size, out)

    def step(self, x: Tensor, states: States, new: NewStates) -> Tensor:
        for unit in self.residuals:
            x = unit.step(x, states, new)
        return F.elu(self.down.step(x, states, new))


class DecoderBlock(nn.Module):
    """Transposed conv (upsample by `stride`), then residual units at the new rate."""

    def __init__(
        self, in_channels: int, out_channels: int, stride: int, residual_units: int,
        kernel_size: int,
    ) -> None:  # fmt: skip
        super().__init__()
        self.up = CausalConvTranspose1d(in_channels, out_channels, kernel_size=2 * stride, stride=stride)
        self.residuals = nn.ModuleList(
            ResidualUnit(out_channels, dilation=3**i, kernel_size=kernel_size) for i in range(residual_units)
        )

    def forward(self, x: Tensor) -> Tensor:
        x = F.elu(self.up(x))
        for unit in self.residuals:
            x = unit(x)
        return x

    def init_state(self, batch_size: int, out: NewStates) -> None:
        self.up.init_state(batch_size, out)
        for unit in self.residuals:
            unit.init_state(batch_size, out)

    def step(self, x: Tensor, states: States, new: NewStates) -> Tensor:
        x = F.elu(self.up.step(x, states, new))
        for unit in self.residuals:
            x = unit.step(x, states, new)
        return x


class FiLM(nn.Module):
    """Feature-wise linear modulation: x * (1 + gamma(c)) + beta(c), per channel.

    Predicting 1 + gamma (not gamma) means a fresh layer starts as the identity
    instead of multiplying the signal by a random number.
    Paper: "FiLM: Visual Reasoning with a General Conditioning Layer".
    """

    def __init__(self, channels: int, cond_dim: int) -> None:
        super().__init__()
        self.to_gamma = nn.Linear(cond_dim, channels)
        self.to_beta = nn.Linear(cond_dim, channels)

    def forward(self, x: Tensor, cond: Tensor) -> Tensor:
        # x [B, C, T], cond [B, cond_dim] -> gamma, beta [B, C, 1]
        gamma = self.to_gamma(cond).unsqueeze(-1)
        beta = self.to_beta(cond).unsqueeze(-1)
        return x * (1.0 + gamma) + beta


# ---------------------------------------------------------------------------
# The generator
# ---------------------------------------------------------------------------


def init_weights(module: nn.Module) -> None:
    """Small normal init for convs (as HiFi-GAN), Xavier for linear layers."""
    if isinstance(module, (nn.Conv1d, nn.ConvTranspose1d)):
        nn.init.normal_(module.weight, 0.0, 0.01)
        nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)


class PairVC(nn.Module):
    """Any-to-N voice converter: source speech + target speaker id -> converted speech."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.grad_checkpointing = False  # set by train.py; saves memory, costs compute
        ch, k = config.channels, config.kernel_size
        widths = [ch * 2**i for i in range(len(config.strides) + 1)]  # 32, 64, 128, 256, 512

        # encoder: 1 -> 32 -> ... -> 512 channels, T -> T / hop frames
        self.enc_in = CausalConv1d(1, widths[0], k)
        self.enc_blocks = nn.ModuleList(
            EncoderBlock(widths[i], widths[i + 1], s, config.residual_units, k)
            for i, s in enumerate(config.strides)
        )
        self.enc_out = CausalConv1d(widths[-1], config.latent_dim, kernel_size=3)

        # decoder: mirror of the encoder, each block followed by speaker FiLM
        self.speaker_embedding = nn.Embedding(config.n_speakers, config.speaker_dim)
        self.dec_in = CausalConv1d(config.latent_dim, widths[-1], k)
        n = len(config.strides)
        self.dec_blocks = nn.ModuleList(
            DecoderBlock(widths[n - i], widths[n - i - 1], s, config.residual_units, k)
            for i, s in enumerate(reversed(config.strides))
        )
        self.films = nn.ModuleList(FiLM(widths[n - i - 1], config.speaker_dim) for i in range(n))
        self.dec_out = CausalConv1d(widths[0], 1, k)

        self.apply(init_weights)

    # --- whole clip -----------------------------------------------------------

    def _block(self, block: nn.Module, x: Tensor) -> Tensor:
        if self.grad_checkpointing and self.training:
            # Recompute the block's activations in backward instead of storing them.
            return checkpoint(block, x, use_reentrant=False)
        return block(x)

    def encode(self, audio: Tensor) -> Tensor:
        """[B, T] audio -> [B, latent_dim, T / hop] latent frames."""
        x = F.elu(self.enc_in(audio.unsqueeze(1)))
        for block in self.enc_blocks:
            x = self._block(block, x)
        return self.enc_out(x)

    def decode(self, latent: Tensor, speaker: Tensor) -> Tensor:
        """[B, latent_dim, frames] + [B] speaker ids -> [B, frames * hop] audio."""
        cond = self.speaker_embedding(speaker)
        x = F.elu(self.dec_in(latent))
        for block, film in zip(self.dec_blocks, self.films):
            x = film(self._block(block, x), cond)
        return self.dec_out(x).squeeze(1)

    def forward(self, audio: Tensor, speaker: Tensor) -> Tensor:
        """Convert a batch of clips: [B, T] float audio, [B] long speaker ids -> [B, T].

        The output is *delayed* by `config.lookahead_samples` (see `convert`).
        T need not be a multiple of the hop: the end is zero-padded and cut off
        again (padding the end cannot change earlier outputs of a causal model).
        """
        length = audio.shape[-1]
        pad = -length % self.config.hop_length
        if pad:
            audio = F.pad(audio, (0, pad))
        return self.decode(self.encode(audio), speaker)[..., :length]

    @torch.inference_mode()
    def convert(self, audio: Tensor, speaker: Tensor) -> Tensor:
        """Offline conversion, time-aligned with the input: [B, T] -> [B, T].

        The model was trained to output sample t at position t + lookahead, so
        we feed `lookahead` extra zeros and drop the first `lookahead` outputs.
        """
        delay = self.config.lookahead_samples
        out = self(F.pad(audio, (0, delay)), speaker)
        return out[..., delay:]

    # --- streaming --------------------------------------------------------------

    def _stateful_layers(self) -> list[nn.Module]:
        """Every layer with a streaming state, in the order states are stored."""
        return [self.enc_in, *self.enc_blocks, self.enc_out, self.dec_in, *self.dec_blocks,
                self.dec_out]  # fmt: skip

    def init_state(self, batch_size: int = 1) -> list[Tensor]:
        """Zero streaming states: the model has heard nothing yet."""
        states: list[Tensor] = []
        for layer in self._stateful_layers():
            layer.init_state(batch_size, states)
        device = self.enc_in.weight.device
        return [s.to(device) for s in states]

    def stream_step(
        self, chunk: Tensor, speaker: Tensor, states: list[Tensor]
    ) -> tuple[Tensor, list[Tensor]]:
        """Convert one chunk: [B, n * hop] -> [B, n * hop], plus the next states.

        Feeding a clip chunk by chunk gives exactly `forward` on the whole clip
        (the output is delayed by `lookahead_samples`, as in `forward`).
        """
        if chunk.shape[-1] % self.config.hop_length:
            raise ValueError(f"chunk length must be a multiple of {self.config.hop_length}")
        it, new = iter(states), []
        x = F.elu(self.enc_in.step(chunk.unsqueeze(1), it, new))
        for block in self.enc_blocks:
            x = block.step(x, it, new)
        x = self.enc_out.step(x, it, new)

        cond = self.speaker_embedding(speaker)
        x = F.elu(self.dec_in.step(x, it, new))
        for block, film in zip(self.dec_blocks, self.films):
            x = film(block.step(x, it, new), cond)
        x = self.dec_out.step(x, it, new)
        return x.squeeze(1), new

    # --- info ---------------------------------------------------------------------

    def receptive_field(self) -> int:
        """How many input samples can influence one output sample (encoder + decoder).

        Measured, not derived: the gradient of the last output sample is non-zero
        exactly on the input samples it depends on.
        """
        audio = torch.randn(1, self.config.hop_length * 128, requires_grad=True)
        was_training = self.training
        self.eval()
        with torch.enable_grad():
            device = self.enc_in.weight.device
            out = self(audio.to(device), torch.zeros(1, dtype=torch.long, device=device))
            out[0, -1].backward()
        self.train(was_training)
        self.zero_grad(set_to_none=True)
        used = (audio.grad[0] != 0).nonzero()
        return int(audio.shape[-1] - used.min())

### 4.2 Build it and check streaming
Converting a clip chunk by chunk (as a live app does) must give the same audio
as converting it in one go.

In [ ]:
torch.manual_seed(train_config.seed)
generator = PairVC(model_config)
print(f"generator: {sum(p.numel() for p in generator.parameters()) / 1e6:.2f}M parameters, "
      f"hop {model_config.hop_length} samples, lookahead {1000 * model_config.lookahead_samples / data_config.sample_rate:.0f} ms")

with torch.no_grad():
    x, spk = 0.1 * torch.randn(1, model_config.hop_length * 20), torch.tensor([0])
    full = generator.eval()(x, spk)
    states, chunks = generator.init_state(1), []
    for start in range(0, x.shape[-1], model_config.hop_length):
        out, states = generator.stream_step(x[:, start:start + model_config.hop_length], spk, states)
        chunks.append(out)
    error = (torch.cat(chunks, -1) - full).abs().max().item()
generator.train()
print(f"max |streaming - whole clip| = {error:.1e} (float rounding only)")

## 5. Losses & discriminators

### 5.1 Losses (same as `src/pairvc/losses.py`)

Training losses.

Generator loss = lambda_adv * adversarial + lambda_fm * feature matching + lambda_mel * mel

- mel (the main signal): L1 between log-mel spectrograms of the generated and
  the teacher audio, at 7 resolutions (5 mels / 32-sample window ... 320 mels /
  2048-sample window), as in DAC. On its own it gives intelligible but buzzy audio.
- adversarial + feature matching (least-squares GAN, as HiFi-GAN): make the
  audio sound natural; feature matching = L1 between the discriminators'
  hidden activations on real and generated audio.

In [ ]:
import math
import warnings

import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn

# ---------------------------------------------------------------------------
# GAN losses (least squares)
# ---------------------------------------------------------------------------


def discriminator_loss(real_scores: list[Tensor], fake_scores: list[Tensor]) -> tuple[Tensor, Tensor, Tensor]:
    """D wants real -> 1 and fake -> 0. Returns (loss, mean real term, mean fake term)."""
    real = torch.stack([torch.mean((1 - r.float()) ** 2) for r in real_scores]).mean()
    fake = torch.stack([torch.mean(f.float() ** 2) for f in fake_scores]).mean()
    return real + fake, real.detach(), fake.detach()


def generator_adversarial_loss(fake_scores: list[Tensor]) -> Tensor:
    """G wants the discriminators to score its audio as real (1)."""
    return torch.stack([torch.mean((1 - f.float()) ** 2) for f in fake_scores]).mean()


def feature_matching_loss(real_fmaps: list[list[Tensor]], fake_fmaps: list[list[Tensor]]) -> Tensor:
    """Mean L1 between hidden activations; the final score layer of each D is left out."""
    losses = [
        torch.mean(torch.abs(f.float() - r.detach().float()))
        for real, fake in zip(real_fmaps, fake_fmaps, strict=True)
        for r, f in zip(real[:-1], fake[:-1], strict=True)
    ]
    return torch.stack(losses).mean()


# ---------------------------------------------------------------------------
# Multi-scale mel loss
# ---------------------------------------------------------------------------


def _hz_to_mel(hz: np.ndarray) -> np.ndarray:
    """Slaney mel scale (librosa's default): linear below 1 kHz, log above."""
    hz = np.asarray(hz, dtype=np.float64)
    f_sp, min_log_hz = 200.0 / 3, 1000.0
    min_log_mel, logstep = min_log_hz / f_sp, math.log(6.4) / 27.0
    return np.where(
        hz < min_log_hz, hz / f_sp, min_log_mel + np.log(np.maximum(hz, 1e-10) / min_log_hz) / logstep
    )


def _mel_to_hz(mel: np.ndarray) -> np.ndarray:
    mel = np.asarray(mel, dtype=np.float64)
    f_sp, min_log_hz = 200.0 / 3, 1000.0
    min_log_mel, logstep = min_log_hz / f_sp, math.log(6.4) / 27.0
    return np.where(mel < min_log_mel, mel * f_sp, min_log_hz * np.exp(logstep * (mel - min_log_mel)))


def mel_filterbank(sample_rate: int, n_fft: int, n_mels: int) -> Tensor:
    """[n_mels, n_fft // 2 + 1] triangular filters, Slaney-normalised (same as librosa.filters.mel)."""
    freqs = np.linspace(0, sample_rate / 2, n_fft // 2 + 1)
    edges = _mel_to_hz(np.linspace(_hz_to_mel(0.0), _hz_to_mel(sample_rate / 2), n_mels + 2))
    widths = np.diff(edges)
    ramps = edges[:, None] - freqs[None, :]
    lower = -ramps[:-2] / widths[:-1, None]
    upper = ramps[2:] / widths[1:, None]
    weights = np.maximum(0, np.minimum(lower, upper))
    weights *= (2.0 / (edges[2:] - edges[:-2]))[:, None]  # equal area per filter
    return torch.from_numpy(weights).float()


class MultiScaleMelLoss(nn.Module):
    """Sum over the 7 scales of L1(log10 mel(x), log10 mel(y))."""

    def __init__(
        self,
        sample_rate: int,
        n_mels: tuple[int, ...] = (5, 10, 20, 40, 80, 160, 320),
        window_lengths: tuple[int, ...] = (32, 64, 128, 256, 512, 1024, 2048),
        clamp_eps: float = 1e-5,
    ) -> None:
        super().__init__()
        self.window_lengths = window_lengths
        self.clamp_eps = clamp_eps
        for i, (mels, win) in enumerate(zip(n_mels, window_lengths, strict=True)):
            # Not saved in checkpoints (persistent=False): they are rebuilt from the numbers above.
            self.register_buffer(f"window_{i}", torch.hann_window(win), persistent=False)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")  # a few top-resolution filters may be empty: harmless
                self.register_buffer(f"mel_{i}", mel_filterbank(sample_rate, win, mels), persistent=False)

    def log_mel(self, audio: Tensor, i: int) -> Tensor:
        win = self.window_lengths[i]
        spec = torch.stft(
            audio.float(), win, hop_length=win // 4, window=getattr(self, f"window_{i}"),
            center=True, pad_mode="reflect", return_complex=True,
        ).abs()  # fmt: skip
        mel = getattr(self, f"mel_{i}") @ spec  # [B, n_mels, frames]
        return torch.log10(mel.clamp(min=self.clamp_eps))

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        """x, y: [B, T] audio (T > 1024 for the largest window)."""
        with torch.autocast(device_type=x.device.type, enabled=False):
            losses = [
                F.l1_loss(self.log_mel(x, i), self.log_mel(y, i)) for i in range(len(self.window_lengths))
            ]
        return torch.stack(losses).sum()

### 5.2 Discriminators (same as `src/pairvc/discriminators.py`)

Discriminators: small networks that learn to tell teacher audio from PairVC audio.

They are only used for training (the GAN part). Each one looks at the audio in
a different way, so the generator cannot fool them all with the same artifact:

- MultiScaleDiscriminator      raw waveform at 1x, 1/2x, 1/4x rate  (HiFi-GAN)
- MultiPeriodDiscriminator     waveform folded by periods 2, 3, 5, 7, 11  (HiFi-GAN)
- MultiResolutionDiscriminator STFT magnitude at 3 resolutions  (UnivNet)
- MultiBandDiscriminator       complex STFT split into frequency bands  (DAC)

Every discriminator returns, for real audio y and generated audio y_hat:
    real_scores, fake_scores: one score map per sub-discriminator
    real_fmaps,  fake_fmaps:  the hidden activations, for feature matching

In [ ]:
from itertools import pairwise

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.nn.utils.parametrizations import spectral_norm, weight_norm


Scores = list[Tensor]
FeatureMaps = list[list[Tensor]]
DiscOutput = tuple[Scores, Scores, FeatureMaps, FeatureMaps]


def stft_fp32(x: Tensor, n_fft: int, hop: int, win: int, center: bool = True) -> Tensor:
    """Complex STFT [B, F, frames] computed in fp32 even under bf16 autocast."""
    with torch.autocast(device_type=x.device.type, enabled=False):
        window = torch.hann_window(win, device=x.device)
        return torch.stft(
            x.float(), n_fft, hop_length=hop, win_length=win, window=window,
            center=center, return_complex=True,
        )  # fmt: skip


class _PairwiseDiscriminator(nn.Module):
    """Runs every sub-discriminator on real and fake audio and collects the outputs."""

    discriminators: nn.ModuleList

    def forward(self, y: Tensor, y_hat: Tensor) -> DiscOutput:
        real_scores, fake_scores, real_fmaps, fake_fmaps = [], [], [], []
        for d in self.discriminators:
            score, fmap = d(y)
            real_scores.append(score)
            real_fmaps.append(fmap)
            score, fmap = d(y_hat)
            fake_scores.append(score)
            fake_fmaps.append(fmap)
        return real_scores, fake_scores, real_fmaps, fake_fmaps


# ---------------------------------------------------------------------------
# Waveform discriminators
# ---------------------------------------------------------------------------


class ScaleDiscriminator(nn.Module):
    """Strided grouped 1D convs over the raw waveform [B, 1, T]."""

    def __init__(self, lrelu_slope: float, use_spectral_norm: bool = False) -> None:
        super().__init__()
        norm = spectral_norm if use_spectral_norm else weight_norm
        self.lrelu_slope = lrelu_slope
        self.convs = nn.ModuleList([
            norm(nn.Conv1d(1, 16, 15, 1, padding=7)),
            norm(nn.Conv1d(16, 64, 41, 4, groups=4, padding=20)),
            norm(nn.Conv1d(64, 256, 41, 4, groups=16, padding=20)),
            norm(nn.Conv1d(256, 1024, 41, 4, groups=64, padding=20)),
            norm(nn.Conv1d(1024, 1024, 41, 4, groups=256, padding=20)),
            norm(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])  # fmt: skip
        self.conv_post = norm(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x: Tensor) -> tuple[Tensor, list[Tensor]]:
        fmap = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), self.lrelu_slope)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        return x.flatten(1), fmap


class MultiScaleDiscriminator(_PairwiseDiscriminator):
    """ScaleDiscriminators on the waveform average-pooled 1x, 2x, 4x, ..."""

    def __init__(self, config: DiscConfig) -> None:
        super().__init__()
        # HiFi-GAN: spectral norm on the full-rate one, weight norm on the others.
        self.discriminators = nn.ModuleList(
            ScaleDiscriminator(config.lrelu_slope, use_spectral_norm=i == 0)
            for i in range(config.time_scales)
        )
        self.pool = nn.AvgPool1d(kernel_size=4, stride=2, padding=2)

    def forward(self, y: Tensor, y_hat: Tensor) -> DiscOutput:
        real_scores, fake_scores, real_fmaps, fake_fmaps = [], [], [], []
        for i, d in enumerate(self.discriminators):
            if i > 0:
                y, y_hat = self.pool(y), self.pool(y_hat)
            for audio, scores, fmaps in ((y, real_scores, real_fmaps), (y_hat, fake_scores, fake_fmaps)):
                score, fmap = d(audio)
                scores.append(score)
                fmaps.append(fmap)
        return real_scores, fake_scores, real_fmaps, fake_fmaps


class PeriodDiscriminator(nn.Module):
    """Folds [B, 1, T] into [B, 1, T/p, p] and applies (k x 1) 2D convs."""

    def __init__(self, period: int, kernel_size: int, stride: int, lrelu_slope: float) -> None:
        super().__init__()
        self.period = period
        self.lrelu_slope = lrelu_slope
        pad = (kernel_size - 1) // 2
        channels = [1, 32, 128, 512, 1024, 1024]
        self.convs = nn.ModuleList(
            weight_norm(nn.Conv2d(
                channels[i], channels[i + 1], (kernel_size, 1),
                (stride if i < 4 else 1, 1), padding=(pad, 0),
            ))
            for i in range(5)
        )  # fmt: skip
        self.conv_post = weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x: Tensor) -> tuple[Tensor, list[Tensor]]:
        b, c, t = x.shape
        if t % self.period:
            x = F.pad(x, (0, self.period - t % self.period), mode="reflect")
        x = x.view(b, c, -1, self.period)
        fmap = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), self.lrelu_slope)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        return x.flatten(1), fmap


class MultiPeriodDiscriminator(_PairwiseDiscriminator):
    def __init__(self, config: DiscConfig) -> None:
        super().__init__()
        self.discriminators = nn.ModuleList(
            PeriodDiscriminator(p, config.period_kernel_size, config.period_stride, config.lrelu_slope)
            for p in config.periods
        )


# ---------------------------------------------------------------------------
# Spectrogram discriminators
# ---------------------------------------------------------------------------


class ResolutionDiscriminator(nn.Module):
    """2D convs over one STFT magnitude [B, 1, F, frames]."""

    def __init__(self, resolution: tuple[int, int, int], channels: int, lrelu_slope: float) -> None:
        super().__init__()
        self.n_fft, self.hop, self.win = resolution
        self.lrelu_slope = lrelu_slope
        c = channels
        self.convs = nn.ModuleList([
            weight_norm(nn.Conv2d(1, c, (3, 9), padding=(1, 4))),
            weight_norm(nn.Conv2d(c, c, (3, 9), stride=(1, 2), padding=(1, 4))),
            weight_norm(nn.Conv2d(c, c, (3, 9), stride=(1, 2), padding=(1, 4))),
            weight_norm(nn.Conv2d(c, c, (3, 9), stride=(1, 2), padding=(1, 4))),
            weight_norm(nn.Conv2d(c, c, (3, 3), padding=(1, 1))),
        ])  # fmt: skip
        self.conv_post = weight_norm(nn.Conv2d(c, 1, (3, 3), padding=(1, 1)))

    def forward(self, x: Tensor) -> tuple[Tensor, list[Tensor]]:
        pad = (self.n_fft - self.hop) // 2
        x = F.pad(x, (pad, pad), mode="reflect").squeeze(1)
        # center=False: we already padded (padding twice shifts the frames).
        x = stft_fp32(x, self.n_fft, self.hop, self.win, center=False).abs().unsqueeze(1)
        fmap = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), self.lrelu_slope)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        return x.flatten(1), fmap


class MultiResolutionDiscriminator(_PairwiseDiscriminator):
    def __init__(self, config: DiscConfig) -> None:
        super().__init__()
        self.discriminators = nn.ModuleList(
            ResolutionDiscriminator(r, config.resolution_channels, config.lrelu_slope)
            for r in config.resolutions
        )


class BandDiscriminator(nn.Module):
    """Complex STFT (real + imag as 2 channels) split into frequency bands, one conv stack each."""

    def __init__(self, n_fft: int, config: DiscConfig) -> None:
        super().__init__()
        self.n_fft = n_fft
        self.lrelu_slope = config.lrelu_slope
        n_bins = n_fft // 2 + 1
        edges = [int(e * n_bins) for e in config.band_edges]
        self.bands = list(pairwise(edges))
        c = config.band_channels

        def stack() -> nn.ModuleList:
            return nn.ModuleList([
                weight_norm(nn.Conv2d(2, c, (3, 9), (1, 1), padding=(1, 4))),
                weight_norm(nn.Conv2d(c, c, (3, 9), (1, 2), padding=(1, 4))),
                weight_norm(nn.Conv2d(c, c, (3, 9), (1, 2), padding=(1, 4))),
                weight_norm(nn.Conv2d(c, c, (3, 9), (1, 2), padding=(1, 4))),
                weight_norm(nn.Conv2d(c, c, (3, 3), (1, 1), padding=(1, 1))),
            ])  # fmt: skip

        self.band_convs = nn.ModuleList(stack() for _ in self.bands)
        self.conv_post = weight_norm(nn.Conv2d(c, 1, (3, 3), (1, 1), padding=(1, 1)))

    def forward(self, x: Tensor) -> tuple[Tensor, list[Tensor]]:
        x = x.squeeze(1)
        x = x - x.mean(dim=-1, keepdim=True)  # remove DC offset
        x = 0.8 * x / (x.abs().max(dim=-1, keepdim=True).values + 1e-9)  # peak-normalise
        spec = stft_fp32(x, self.n_fft, self.n_fft // 4, self.n_fft)
        spec = torch.view_as_real(spec).permute(0, 3, 2, 1)  # [B, 2, frames, F]
        fmap, outs = [], []
        for (lo, hi), convs in zip(self.bands, self.band_convs):
            band = spec[..., lo:hi]
            for i, conv in enumerate(convs):
                band = F.leaky_relu(conv(band), self.lrelu_slope)
                if i > 0:
                    fmap.append(band)
            outs.append(band)
        x = self.conv_post(torch.cat(outs, dim=-1))
        fmap.append(x)
        return x.flatten(1), fmap


class MultiBandDiscriminator(_PairwiseDiscriminator):
    def __init__(self, config: DiscConfig) -> None:
        super().__init__()
        self.discriminators = nn.ModuleList(
            BandDiscriminator(n_fft, config) for n_fft in config.band_fft_sizes
        )


# ---------------------------------------------------------------------------
# All of them together
# ---------------------------------------------------------------------------

_KINDS = {
    "time": MultiScaleDiscriminator,
    "period": MultiPeriodDiscriminator,
    "resolution": MultiResolutionDiscriminator,
    "band": MultiBandDiscriminator,
}


class Discriminators(nn.Module):
    """The discriminators named in `config.kinds`, run together; outputs are concatenated."""

    def __init__(self, config: DiscConfig) -> None:
        super().__init__()
        self.config = config
        self.discriminators = nn.ModuleList(_KINDS[kind](config) for kind in config.kinds)

    def forward(self, y: Tensor, y_hat: Tensor) -> DiscOutput:
        """y (target) and y_hat (generated): [B, T] or [B, 1, T]."""
        if y.dim() == 2:
            y = y.unsqueeze(1)
        if y_hat.dim() == 2:
            y_hat = y_hat.unsqueeze(1)
        real_scores, fake_scores, real_fmaps, fake_fmaps = [], [], [], []
        for d in self.discriminators:
            rs, fs, rf, ff = d(y, y_hat)
            real_scores += rs
            fake_scores += fs
            real_fmaps += rf
            fake_fmaps += ff
        return real_scores, fake_scores, real_fmaps, fake_fmaps

In [ ]:
discriminator = Discriminators(disc_config)
print(f"discriminators {disc_config.kinds}: {sum(p.numel() for p in discriminator.parameters()) / 1e6:.2f}M parameters "
      "(training only, not needed to convert)")

## 6. Training code (same as `src/pairvc/train.py`)

Train PairVC on paired (source, teacher-converted) audio.

The file is split into small pieces you can read top to bottom:

1. setup helpers   - seeding, device, mixed precision, multi-GPU (DDP) setup
2. data            - DataLoaders
3. optimisation    - two AdamW optimizers (generator, discriminators) + LR decay
4. one GAN step    - discriminator update, then generator update
5. evaluation      - validation mel loss, one audio sample to listen to
6. checkpoints     - save / load everything needed to resume
7. train()         - the training loop (takes ready-made objects, easy to test)
8. CLI             - main() builds everything from command-line flags

Run it with:  python -m pairvc.train --help

Multi-GPU / multi-node: launch the same command with torchrun, e.g.
    torchrun --nproc_per_node=4 -m pairvc.train ...
torchrun starts one process per GPU and sets RANK / WORLD_SIZE / LOCAL_RANK;
the code below picks those up and wraps both networks in DistributedDataParallel.

The command-line part is included but not used: section 8 calls `train()`
directly with the objects built above.

In [ ]:
import argparse
import dataclasses
import json
import logging
import math
import os
import random
import time
from collections.abc import Callable, Iterator, Sequence
from contextlib import AbstractContextManager
from dataclasses import dataclass
from itertools import count, islice
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch
import torch.distributed as dist
from torch import Tensor, nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW, Optimizer
from torch.optim.lr_scheduler import ExponentialLR
from torch.utils.data import DataLoader, Dataset, DistributedSampler


logger = logging.getLogger(__name__)

LAST_CHECKPOINT = "last.pt"
BEST_CHECKPOINT = "best.pt"
HISTORY_FILE = "history.jsonl"  # every logged train / val event, one JSON object per line


# ---------------------------------------------------------------------------
# 1. Setup helpers
# ---------------------------------------------------------------------------


def seed_everything(seed: int) -> None:
    """Seed every random number generator we use."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds all CUDA devices


def resolve_device(name: str = "auto") -> torch.device:
    """Turn "auto" | "cpu" | "cuda" | "mps" into a torch.device."""
    # Under torchrun each process owns one GPU: cuda:LOCAL_RANK.
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if name == "cuda" or (name == "auto" and torch.cuda.is_available()):
        return torch.device("cuda", local_rank)
    if name != "auto":
        return torch.device(name)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def resolve_precision(precision: str, device: torch.device) -> torch.dtype | None:
    """Autocast dtype, or None for plain fp32.

    Only bf16 is offered: it has the range of fp32, so the GAN losses cannot
    overflow and no GradScaler is needed. GPUs without native bf16 (T4, P100,
    V100) train in fp32.
    """
    if precision == "fp32":
        return None
    if precision == "bf16":
        return torch.bfloat16
    if precision == "auto":
        if device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False):
            return torch.bfloat16
        return None
    raise ValueError(f"unknown precision: {precision!r}")


def autocast(device: torch.device, dtype: torch.dtype | None) -> AbstractContextManager:
    """Mixed-precision context; does nothing when dtype is None."""
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=dtype is not None)


@dataclass(frozen=True)
class DistInfo:
    """Who am I in a multi-process run? Defaults describe a single process."""

    rank: int = 0  # global id of this process (0 .. world_size-1)
    world_size: int = 1  # total number of processes (= total GPUs)
    local_rank: int = 0  # id of this process on its own machine

    @property
    def enabled(self) -> bool:
        return self.world_size > 1

    @property
    def is_main(self) -> bool:
        """Only the main process logs, writes audio samples and checkpoints."""
        return self.rank == 0


def setup_distributed(device_name: str = "auto") -> DistInfo:
    """Join the process group if we were launched by torchrun; no-op otherwise."""
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    if world_size == 1:
        return DistInfo()
    local_rank = int(os.environ["LOCAL_RANK"])
    use_cuda = device_name in ("auto", "cuda") and torch.cuda.is_available()
    if use_cuda:
        torch.cuda.set_device(local_rank)  # this process only talks to its own GPU
    # NCCL is the fast backend for GPUs; gloo works on CPU (handy for testing).
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


def get_dist_info() -> DistInfo:
    """Current process-group state (single process if none was started)."""
    if not (dist.is_available() and dist.is_initialized()):
        return DistInfo()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


# ---------------------------------------------------------------------------
# 2. Data
# ---------------------------------------------------------------------------


def seed_worker(worker_id: int) -> None:
    """Give each DataLoader worker its own python/numpy seed (random crops differ)."""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def build_dataloaders(
    train_dataset: Dataset, val_dataset: Dataset, config: TrainConfig, device: torch.device
) -> tuple[DataLoader, DataLoader]:
    """Create the train and validation DataLoaders.

    With several processes, a DistributedSampler gives each one a different
    slice of the data, so batch_size is *per process* (per GPU).
    """
    info = get_dist_info()
    workers = config.num_workers
    common = {
        "batch_size": config.batch_size,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",  # faster CPU -> GPU copies
        "persistent_workers": workers > 0,  # do not restart workers every epoch
        "worker_init_fn": seed_worker if workers > 0 else None,
    }
    generator = torch.Generator()
    generator.manual_seed(config.seed + info.rank)  # + rank: different crops per process

    if info.enabled:
        train_sampler = DistributedSampler(train_dataset, shuffle=True, seed=config.seed, drop_last=True)
        val_sampler = DistributedSampler(val_dataset, shuffle=False, drop_last=False)
        train_loader = DataLoader(
            train_dataset, sampler=train_sampler, drop_last=True, generator=generator, **common
        )
        val_loader = DataLoader(val_dataset, sampler=val_sampler, drop_last=False, **common)
        return train_loader, val_loader

    train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, generator=generator, **common)
    val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader


def move_batch(batch: dict[str, Tensor], device: torch.device) -> dict[str, Tensor]:
    """Send every tensor of the batch to the device."""
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


# ---------------------------------------------------------------------------
# 3. Optimisation
# ---------------------------------------------------------------------------


def build_optimizers(
    generator: nn.Module, discriminator: nn.Module, config: TrainConfig
) -> tuple[AdamW, AdamW]:
    """One AdamW per network: they are updated in turn, never together."""
    kwargs = {"betas": config.betas, "weight_decay": config.weight_decay}
    return (
        AdamW(generator.parameters(), lr=config.g_lr, **kwargs),
        AdamW(discriminator.parameters(), lr=config.d_lr, **kwargs),
    )


def build_schedulers(
    optim_g: Optimizer, optim_d: Optimizer, config: TrainConfig
) -> tuple[ExponentialLR, ExponentialLR]:
    """LR *= lr_decay after every step (0.9999996 -> x0.82 after 500k steps)."""
    return ExponentialLR(optim_g, config.lr_decay), ExponentialLR(optim_d, config.lr_decay)


def clip_gradients(model: nn.Module, max_norm: float) -> Tensor:
    """Total gradient norm (before clipping); clips only if max_norm > 0."""
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm if max_norm > 0 else math.inf)


# ---------------------------------------------------------------------------
# 4. One GAN step
# ---------------------------------------------------------------------------


def align_with_lookahead(prediction: Tensor, target: Tensor, lookahead: int) -> tuple[Tensor, Tensor]:
    """Compare output sample t + lookahead with target sample t.

    The causal model may hear `lookahead` samples of future source audio before
    it must produce target sample t: a little latency, much better quality.
    """
    if lookahead == 0:
        return prediction, target
    return prediction[..., lookahead:], target[..., : target.shape[-1] - lookahead]


def gan_step(
    generator: nn.Module,
    discriminator: nn.Module,
    raw_discriminator: nn.Module,
    batch: dict[str, Tensor],
    mel_loss: MultiScaleMelLoss,
    optim_g: Optimizer,
    optim_d: Optimizer,
    config: TrainConfig,
    lookahead: int,
    amp_dtype: torch.dtype | None,
    use_gan: bool = True,
) -> dict[str, Tensor]:
    """One discriminator update then one generator update. Returns detached metrics.

    With `use_gan=False` (the mel-only warm-up, see TrainConfig.gan_start_step)
    the discriminators are skipped and the generator learns from the mel loss alone.

    `discriminator` may be DDP-wrapped (its update is all-reduced);
    `raw_discriminator` is the same network unwrapped, used in the generator
    update with its weights frozen: there we only need gradients w.r.t. the
    *generated audio*, and going through DDP would wait for gradients that
    never come.
    """
    device = batch["source"].device
    with autocast(device, amp_dtype):
        prediction = generator(batch["source"], batch["speaker"])
    prediction, target = align_with_lookahead(prediction.float(), batch["target"], lookahead)
    if not use_gan:
        mel = mel_loss(prediction, target)
        optim_g.zero_grad(set_to_none=True)
        (config.lambda_mel * mel).backward()
        g_grad = clip_gradients(generator, config.grad_clip)
        optim_g.step()
        zero = torch.zeros((), device=device)
        return {
            "d_loss": zero, "d_real": zero, "d_fake": zero, "g_loss": config.lambda_mel * mel.detach(),
            "adv": zero, "fm": zero, "mel": mel.detach(), "g_grad": g_grad.detach(), "d_grad": zero,
        }  # fmt: skip

    # --- discriminators: real (teacher) -> 1, fake (PairVC) -> 0 ---
    with autocast(device, amp_dtype):
        real_scores, fake_scores, _, _ = discriminator(target, prediction.detach())
    d_loss, d_real, d_fake = discriminator_loss(real_scores, fake_scores)
    optim_d.zero_grad(set_to_none=True)
    d_loss.backward()
    d_grad = clip_gradients(raw_discriminator, config.grad_clip)
    optim_d.step()

    # --- generator: fool the discriminators + sound like the teacher ---
    raw_discriminator.requires_grad_(False)
    try:
        with autocast(device, amp_dtype):
            _, fake_scores, real_fmaps, fake_fmaps = raw_discriminator(target, prediction)
        adv = generator_adversarial_loss(fake_scores)
        fm = feature_matching_loss(real_fmaps, fake_fmaps)
        mel = mel_loss(prediction, target)
        g_loss = config.lambda_adv * adv + config.lambda_fm * fm + config.lambda_mel * mel
        optim_g.zero_grad(set_to_none=True)
        g_loss.backward()
    finally:
        raw_discriminator.requires_grad_(True)
    g_grad = clip_gradients(generator, config.grad_clip)
    optim_g.step()

    return {
        "d_loss": d_loss.detach(), "d_real": d_real, "d_fake": d_fake,
        "g_loss": g_loss.detach(), "adv": adv.detach(), "fm": fm.detach(), "mel": mel.detach(),
        "g_grad": g_grad.detach(), "d_grad": d_grad.detach(),
    }  # fmt: skip


# ---------------------------------------------------------------------------
# 5. Evaluation
# ---------------------------------------------------------------------------


@torch.inference_mode()
def evaluate(
    generator: nn.Module,
    loader: DataLoader,
    mel_loss: MultiScaleMelLoss,
    lookahead: int,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
) -> dict[str, float]:
    """Mean mel loss and waveform L1 against the teacher, over a whole loader.

    The mel loss is the number to watch: lower = closer to the teacher.
    (Adversarial losses go up and down by design and say little on their own.)
    """
    was_training = generator.training
    generator.eval()
    sums = torch.zeros(3, device=device, dtype=torch.float64)  # mel, l1, n_clips
    for batch in loader:
        batch = move_batch(batch, device)
        with autocast(device, amp_dtype):
            prediction = generator(batch["source"], batch["speaker"])
        prediction, target = align_with_lookahead(prediction.float(), batch["target"], lookahead)
        n = target.shape[0]
        sums[0] += mel_loss(prediction, target).double() * n
        sums[1] += (prediction - target).abs().mean().double() * n
        sums[2] += n
    generator.train(was_training)
    if dist.is_initialized():
        dist.all_reduce(sums)  # every process saw a different slice: add them up
    n = sums[2].clamp(min=1)
    return {"mel": (sums[0] / n).item(), "l1": (sums[1] / n).item()}


@torch.inference_mode()
def save_audio_sample(
    generator: nn.Module, dataset: Dataset, path: Path, sample_rate: int, lookahead: int
) -> None:
    """Write source | PairVC | teacher for the first validation clip, one after the other."""
    was_training = generator.training
    generator.eval()
    item = dataset[0]
    device = next(generator.parameters()).device
    prediction = generator(item["source"][None].to(device), item["speaker"][None].to(device))[0]
    generator.train(was_training)
    prediction = prediction[lookahead:].float().cpu()  # time-align with the source
    gap = torch.zeros(sample_rate // 2)
    audio = torch.cat([item["source"], gap, prediction, gap, item["target"]]).clamp(-1, 1)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, audio.numpy(), sample_rate)


# ---------------------------------------------------------------------------
# 6. Checkpoints
# ---------------------------------------------------------------------------


@dataclass
class TrainState:
    """Counters that must survive a restart."""

    step: int = 0  # GAN steps done
    epoch: int = 0
    best_val_mel: float = math.inf


def rng_state() -> dict[str, Any]:
    """Snapshot every RNG so a resumed run continues the same random stream."""
    name, keys, pos, has_gauss, cached = np.random.get_state()
    state: dict[str, Any] = {
        "python": random.getstate(),
        "torch": torch.get_rng_state(),
        # plain lists so torch.load(weights_only=True) can read them back
        "numpy": (name, keys.tolist(), pos, has_gauss, cached),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict[str, Any]) -> None:
    """Restore what rng_state() saved."""
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    name, keys, pos, has_gauss, cached = state["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(
    path: Path,
    generator: nn.Module,
    discriminator: nn.Module,
    optimizers: tuple[Optimizer, Optimizer],
    schedulers: tuple[ExponentialLR, ExponentialLR],
    state: TrainState,
    configs: dict[str, Any],
) -> None:
    """Save everything needed to resume (and to rebuild the model). Written atomically."""
    checkpoint = {
        "generator": generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "optim_g": optimizers[0].state_dict(),
        "optim_d": optimizers[1].state_dict(),
        "sched_g": schedulers[0].state_dict(),
        "sched_d": schedulers[1].state_dict(),
        "step": state.step,
        "epoch": state.epoch,
        "best_val_mel": state.best_val_mel,
        **{f"{name}_config": config_to_dict(cfg) for name, cfg in configs.items()},
        "rng_state": rng_state(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, tmp_path)
    os.replace(tmp_path, path)  # atomic: a crash mid-save never leaves a half-written file


def load_checkpoint(
    path: Path,
    generator: nn.Module,
    discriminator: nn.Module | None = None,
    optimizers: tuple[Optimizer, Optimizer] | None = None,
    schedulers: tuple[ExponentialLR, ExponentialLR] | None = None,
) -> TrainState:
    """Load a checkpoint into the given objects and return the saved counters."""
    # weights_only=True refuses to run arbitrary pickled code (safe loading).
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    generator.load_state_dict(checkpoint["generator"])
    if discriminator is not None:
        discriminator.load_state_dict(checkpoint["discriminator"])
    if optimizers is not None:
        optimizers[0].load_state_dict(checkpoint["optim_g"])
        optimizers[1].load_state_dict(checkpoint["optim_d"])
    if schedulers is not None:
        schedulers[0].load_state_dict(checkpoint["sched_g"])
        schedulers[1].load_state_dict(checkpoint["sched_d"])
    set_rng_state(checkpoint["rng_state"])
    return TrainState(checkpoint["step"], checkpoint["epoch"], checkpoint["best_val_mel"])


# ---------------------------------------------------------------------------
# 7. Training loop
# ---------------------------------------------------------------------------


def epoch_batches(loader: DataLoader, epoch: int, seed: int, skip: int) -> Iterator[dict]:
    """Batches of one epoch, in a shuffle order that depends only on (seed, epoch).

    Re-seeding per epoch means a resumed run sees the same order, so we can
    skip the batches that were already used before the restart.
    """
    if isinstance(loader.sampler, DistributedSampler):
        loader.sampler.set_epoch(epoch)  # new shuffle each epoch, same on every process
    elif loader.generator is not None:
        loader.generator.manual_seed(seed + epoch)
    return islice(iter(loader), skip, None)


def train(
    generator: nn.Module,
    discriminator: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    data_config: DataConfig,
    model_config: ModelConfig,
    disc_config: DiscConfig,
    train_config: TrainConfig,
    callback: Callable[[dict[str, Any]], None] | None = None,
) -> TrainState:
    """Run the GAN training loop for `max_steps` steps and return the final counters.

    Everything is passed in already built, so tests can use tiny datasets.
    `callback` (optional) receives every logged metric as a dict, on the main
    process only; the notebook uses it to draw live training curves, and to stop
    at the session time limit by raising from a "val" event (sent after saving):
        {"kind": "train", "step", "epoch", "mel", "adv", "fm", "d_loss", "d_real",
         "d_fake", "g_grad", "d_grad", "lr_g", "lr_d", "audio_sec_per_sec"}
        {"kind": "val", "step", "epoch", "mel", "l1", "sample_path"}
    """
    cfg = train_config
    info = get_dist_info()
    device = resolve_device(cfg.device)
    amp_dtype = resolve_precision(cfg.precision, device)
    generator.to(device).train()
    discriminator.to(device).train()
    generator.grad_checkpointing = cfg.grad_checkpointing
    raw_g, raw_d = generator, discriminator  # unwrapped: checkpoints, eval, audio samples
    lookahead = model_config.lookahead_samples
    mel_loss = MultiScaleMelLoss(data_config.sample_rate).to(device)

    steps_per_epoch = len(train_loader)
    if steps_per_epoch == 0:
        raise ValueError("train set smaller than one batch: lower --batch-size")

    optimizers = build_optimizers(raw_g, raw_d, cfg)
    schedulers = build_schedulers(*optimizers, cfg)
    configs = {"data": data_config, "model": model_config, "disc": disc_config, "train": cfg}
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    state = TrainState()
    if cfg.resume is not None:
        state = load_checkpoint(Path(cfg.resume), raw_g, raw_d, optimizers, schedulers)
        logger.info("resumed from %s at step %d", cfg.resume, state.step)

    if info.enabled:
        # DDP averages gradients over all processes after each backward pass and
        # copies rank 0's weights to everyone at start, so all begin equal.
        ids = [device.index] if device.type == "cuda" else None
        generator = DDP(generator, device_ids=ids)
        discriminator = DDP(discriminator, device_ids=ids)

    def checkpoint(path: Path) -> None:
        if info.is_main:  # one writer is enough: all processes hold the same weights
            save_checkpoint(path, raw_g, raw_d, optimizers, schedulers, state, configs)

    def emit(event: dict[str, Any]) -> None:
        if not info.is_main:
            return
        with open(output_dir / HISTORY_FILE, "a") as f:  # for plots after the run
            f.write(json.dumps(event) + "\n")
        if callback is not None:
            callback(event)

    def validate_and_save() -> None:
        metrics = evaluate(raw_g, val_loader, mel_loss, lookahead, device, amp_dtype)
        sample_path = output_dir / "samples" / f"step_{state.step:07d}.wav"
        if info.is_main:
            logger.info("val | step %d | mel %.4f | l1 %.4f", state.step, metrics["mel"], metrics["l1"])
            save_audio_sample(raw_g, val_loader.dataset, sample_path, data_config.sample_rate, lookahead)
        if metrics["mel"] < state.best_val_mel:
            state.best_val_mel = metrics["mel"]
            checkpoint(output_dir / BEST_CHECKPOINT)
        checkpoint(output_dir / LAST_CHECKPOINT)
        # Emitted after saving, so a callback may stop training here without losing work.
        emit({"kind": "val", "step": state.step, "epoch": state.epoch, **metrics,
              "sample_path": str(sample_path)})  # fmt: skip

    logger.info(
        "training: %d steps (%d per epoch), batch %d per process x %d processes, "
        "%.1f s clips, on %s, amp=%s",
        cfg.max_steps, steps_per_epoch, cfg.batch_size, info.world_size,
        data_config.segment_samples / data_config.sample_rate, device, amp_dtype,
    )  # fmt: skip

    window: dict[str, Tensor] = {}  # running sums on device: no GPU sync every step
    window_start = time.perf_counter()
    for epoch in count(state.step // steps_per_epoch):
        if state.step >= cfg.max_steps:
            break
        state.epoch = epoch
        done_in_epoch = state.step - epoch * steps_per_epoch  # > 0 only right after resume
        batches = epoch_batches(train_loader, epoch, cfg.seed, skip=done_in_epoch)
        for batch in islice(batches, cfg.max_steps - state.step):
            batch = move_batch(batch, device)
            metrics = gan_step(
                generator, discriminator, raw_d, batch, mel_loss, *optimizers, cfg,
                lookahead, amp_dtype, use_gan=state.step >= cfg.gan_start_step,
            )  # fmt: skip
            for scheduler in schedulers:
                scheduler.step()
            state.step += 1
            for key, value in metrics.items():
                window[key] = window.get(key, 0.0) + value

            if state.step % cfg.log_every == 0:
                elapsed = time.perf_counter() - window_start
                means = {k: (v / cfg.log_every).item() for k, v in window.items()}
                audio_seconds = (
                    cfg.log_every * cfg.batch_size * info.world_size
                    * data_config.segment_samples / data_config.sample_rate
                )  # fmt: skip
                lr_g, lr_d = (s.get_last_lr()[0] for s in schedulers)
                logger.info(
                    "train | epoch %d | step %d/%d | mel %.4f | adv %.3f | fm %.3f | "
                    "d %.3f (real %.3f fake %.3f) | grad g %.1f d %.1f | lr %.2e | %.1f audio s/s",
                    epoch, state.step, cfg.max_steps, means["mel"], means["adv"], means["fm"],
                    means["d_loss"], means["d_real"], means["d_fake"], means["g_grad"],
                    means["d_grad"], lr_g, audio_seconds / elapsed,
                )  # fmt: skip
                emit({"kind": "train", "step": state.step, "epoch": epoch, **means,
                      "lr_g": lr_g, "lr_d": lr_d,
                      "audio_sec_per_sec": audio_seconds / elapsed})  # fmt: skip
                window.clear()
                window_start = time.perf_counter()

            if state.step % cfg.eval_every == 0 or state.step == cfg.max_steps:
                validate_and_save()
                window_start = time.perf_counter()  # don't count eval time in the speed

    logger.info("done | best val mel %.4f", state.best_val_mel)
    return state


# ---------------------------------------------------------------------------
# 8. Command line
# ---------------------------------------------------------------------------


def build_parser() -> argparse.ArgumentParser:
    """All CLI flags. Defaults come from the config dataclasses."""
    d, m, dc, t = DataConfig(), ModelConfig(), DiscConfig(), TrainConfig()
    p = argparse.ArgumentParser(description="Train PairVC on paired voice-conversion data.")

    data = p.add_argument_group("data")
    data.add_argument("--data-dir", type=Path, default=d.data_dir, help="contains train/ and val/")
    data.add_argument("--train-split", default=d.train_split)
    data.add_argument("--val-split", default=d.val_split)
    data.add_argument("--sample-rate", type=int, default=d.sample_rate)
    data.add_argument("--segment-samples", type=int, default=d.segment_samples)

    model = p.add_argument_group("model")
    model.add_argument("--n-speakers", type=int, default=m.n_speakers)
    model.add_argument("--channels", type=int, default=m.channels)
    model.add_argument("--latent-dim", type=int, default=m.latent_dim)
    model.add_argument("--residual-units", type=int, default=m.residual_units)
    model.add_argument("--lookahead-frames", type=int, default=m.lookahead_frames)
    model.add_argument("--discriminators", nargs="+", choices=DISCRIMINATOR_KINDS, default=list(dc.kinds))

    opt = p.add_argument_group("training")
    opt.add_argument("--max-steps", type=int, default=t.max_steps)
    opt.add_argument("--batch-size", type=int, default=t.batch_size)
    opt.add_argument("--g-lr", type=float, default=t.g_lr)
    opt.add_argument("--d-lr", type=float, default=t.d_lr)
    opt.add_argument("--betas", type=float, nargs=2, default=list(t.betas))
    opt.add_argument("--weight-decay", type=float, default=t.weight_decay)
    opt.add_argument("--lr-decay", type=float, default=t.lr_decay)
    opt.add_argument("--grad-clip", type=float, default=t.grad_clip)
    opt.add_argument("--lambda-adv", type=float, default=t.lambda_adv)
    opt.add_argument("--lambda-fm", type=float, default=t.lambda_fm)
    opt.add_argument("--lambda-mel", type=float, default=t.lambda_mel)
    opt.add_argument(
        "--gan-start-step", type=int, default=t.gan_start_step,
        help="mel loss only (no discriminators) before this step: a fast warm-up",
    )  # fmt: skip
    opt.add_argument("--precision", choices=["auto", "fp32", "bf16"], default=t.precision)
    opt.add_argument("--grad-checkpointing", action="store_true")
    opt.add_argument("--num-workers", type=int, default=t.num_workers)
    opt.add_argument("--train-limit", type=int, default=t.train_limit)
    opt.add_argument("--val-limit", type=int, default=t.val_limit)
    opt.add_argument("--log-every", type=int, default=t.log_every)
    opt.add_argument("--eval-every", type=int, default=t.eval_every)
    opt.add_argument("--device", default=t.device)
    opt.add_argument("--output-dir", type=Path, default=t.output_dir)
    opt.add_argument("--resume", type=Path, default=t.resume)
    opt.add_argument("--seed", type=int, default=t.seed)
    opt.add_argument(
        "--overfit",
        type=int,
        metavar="N",
        default=None,
        help="sanity check: train and validate on the same N training pairs",
    )
    return p


def configs_from_args(
    args: argparse.Namespace,
) -> tuple[DataConfig, ModelConfig, DiscConfig, TrainConfig]:
    """Turn parsed flags into the four configs."""
    data_config = DataConfig(
        data_dir=args.data_dir,
        train_split=args.train_split,
        val_split=args.val_split,
        sample_rate=args.sample_rate,
        segment_samples=args.segment_samples,
    )
    model_config = ModelConfig(
        n_speakers=args.n_speakers,
        channels=args.channels,
        latent_dim=args.latent_dim,
        residual_units=args.residual_units,
        lookahead_frames=args.lookahead_frames,
    )
    disc_config = DiscConfig(kinds=tuple(args.discriminators))
    train_config = TrainConfig(
        output_dir=args.output_dir,
        seed=args.seed,
        max_steps=args.max_steps,
        batch_size=args.batch_size,
        g_lr=args.g_lr,
        d_lr=args.d_lr,
        betas=tuple(args.betas),
        weight_decay=args.weight_decay,
        lr_decay=args.lr_decay,
        grad_clip=args.grad_clip,
        lambda_adv=args.lambda_adv,
        lambda_fm=args.lambda_fm,
        lambda_mel=args.lambda_mel,
        gan_start_step=args.gan_start_step,
        precision=args.precision,
        grad_checkpointing=args.grad_checkpointing,
        num_workers=args.num_workers,
        train_limit=args.train_limit,
        val_limit=args.val_limit,
        log_every=args.log_every,
        eval_every=args.eval_every,
        device=args.device,
        resume=args.resume,
    )
    if args.overfit is not None:
        # "Can the model memorise a tiny set?" If not, something is broken.
        n = args.overfit
        data_config = dataclasses.replace(data_config, val_split=data_config.train_split)
        train_config = dataclasses.replace(
            train_config,
            train_limit=n,
            val_limit=n,
            batch_size=min(train_config.batch_size, n),  # drop_last would leave 0 batches
        )
    if data_config.segment_samples % model_config.hop_length:
        raise ValueError(f"--segment-samples must be a multiple of {model_config.hop_length}")
    return data_config, model_config, disc_config, train_config


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point: set up (optional) multi-process training, then train.

    Returns None on purpose: console scripts pass the return value to sys.exit,
    and anything else would be printed and turned into exit code 1.
    """
    args = build_parser().parse_args(argv)
    configs = configs_from_args(args)
    info = setup_distributed(configs[-1].device)  # no-op unless launched by torchrun
    logging.basicConfig(
        level=logging.INFO if info.is_main else logging.WARNING,
        format="%(asctime)s %(levelname)s %(message)s",
        datefmt="%H:%M:%S",
    )
    try:
        _build_and_train(args, *configs)
    finally:
        if info.enabled:
            dist.destroy_process_group()


def _build_and_train(
    args: argparse.Namespace,
    data_config: DataConfig,
    model_config: ModelConfig,
    disc_config: DiscConfig,
    train_config: TrainConfig,
) -> TrainState:
    """Build data and networks from the configs, then train."""
    from pairvc.data import PairedVoiceDataset
    from pairvc.discriminators import Discriminators
    from pairvc.model import PairVC

    seed_everything(train_config.seed)  # before building the networks: same init weights
    root = Path(data_config.data_dir)
    train_dataset = PairedVoiceDataset(
        root / data_config.train_split, data_config.sample_rate, data_config.segment_samples,
        train=args.overfit is None, limit=train_config.train_limit,  # no random crop when memorising
    )  # fmt: skip
    val_dataset = PairedVoiceDataset(
        root / data_config.val_split, data_config.sample_rate, data_config.segment_samples,
        limit=train_config.val_limit,
    )  # fmt: skip
    too_big = [s for s in train_dataset.speakers if s >= model_config.n_speakers]
    if too_big:
        raise ValueError(f"data has target voices {too_big}: set --n-speakers {max(too_big) + 1}")
    logger.info(
        "data: %d train / %d val pairs, target voices %s",
        len(train_dataset), len(val_dataset), train_dataset.speakers,
    )  # fmt: skip

    generator = PairVC(model_config)
    discriminator = Discriminators(disc_config)
    logger.info(
        "generator %.2fM parameters | discriminators %.2fM",
        sum(p.numel() for p in generator.parameters()) / 1e6,
        sum(p.numel() for p in discriminator.parameters()) / 1e6,
    )
    device = resolve_device(train_config.device)
    train_loader, val_loader = build_dataloaders(train_dataset, val_dataset, train_config, device)
    return train(
        generator, discriminator, train_loader, val_loader,
        data_config, model_config, disc_config, train_config,
    )  # fmt: skip

## 7. Live dashboard

- **mel loss** (the one to watch): distance to the teacher's audio in log-mel
  space, train and validation. Lower is better.
- **adversarial / feature matching**: the GAN terms. They do not go steadily
  down: generator and discriminators compete. Worry only if the discriminator
  loss collapses to ~0 for a long time (it wins every time).
- **discriminator real / fake**: (1 - D(real))² and D(fake)²; both should stay
  away from 0.
- **gradient norms**, **learning rate**, **speed** (seconds of audio trained per second).

The dashboard redraws every `log_every` steps; metrics are saved to
`metrics.json` next to the checkpoints, so the curves survive a restart.

In [ ]:
import json
import time

from IPython.display import clear_output

METRICS_FILE = OUTPUT_DIR / "metrics.json"
history = {"train": [], "val": []}
if RESUME and METRICS_FILE.exists() and (OUTPUT_DIR / LAST_CHECKPOINT).exists():
    history = json.loads(METRICS_FILE.read_text())


class TimeBudgetReached(Exception):
    """Raised by the callback after a validation (= after saving) once the time is up."""


def draw_dashboard() -> None:
    train, val = history["train"], history["val"]
    clear_output(wait=True)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    (ax_mel, ax_gan, ax_d), (ax_grad, ax_lr, ax_speed) = axes

    def series(rows, key):
        return [r["step"] for r in rows], [r[key] for r in rows]

    ax_mel.plot(*series(train, "mel"), label="train", alpha=0.8)
    ax_mel.plot(*series(val, "mel"), "o-", label="validation")
    ax_mel.set_title("mel loss vs teacher (lower is better)")
    ax_gan.plot(*series(train, "adv"), label="adversarial")
    ax_gan.plot(*series(train, "fm"), label="feature matching")
    ax_gan.set_title("generator GAN terms")
    ax_d.plot(*series(train, "d_real"), label="real term")
    ax_d.plot(*series(train, "d_fake"), label="fake term")
    ax_d.set_title("discriminator")
    ax_grad.plot(*series(train, "g_grad"), label="generator")
    ax_grad.plot(*series(train, "d_grad"), label="discriminator")
    ax_grad.set_yscale("log")
    ax_grad.set_title("gradient norm")
    ax_lr.plot(*series(train, "lr_g"), label="generator")
    ax_lr.plot(*series(train, "lr_d"), label="discriminator")
    ax_lr.set_title("learning rate")
    ax_speed.plot(*series(train, "audio_sec_per_sec"), color="tab:green")
    ax_speed.set_title("speed (audio seconds / second)")
    for ax in axes.flat:
        ax.set_xlabel("step")
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    fig.tight_layout()
    plt.show()
    if train:
        last = train[-1]
        print(f"step {last['step']:,}/{train_config.max_steps:,} (epoch {last['epoch']}) | "
              f"train mel {last['mel']:.4f} | adv {last['adv']:.3f} | fm {last['fm']:.3f}")
    if val:
        best = min(val, key=lambda r: r["mel"])
        print(f"last validation mel {val[-1]['mel']:.4f} | best {best['mel']:.4f} at step {best['step']:,}")
        print("latest audio sample (source | PairVC | teacher):")
        display(Audio(val[-1]["sample_path"]))


start_time = time.perf_counter()


def on_metrics(event: dict) -> None:
    history[event["kind"]].append(event)
    if event["kind"] == "val":
        METRICS_FILE.write_text(json.dumps(history))
        if (time.perf_counter() - start_time) / 3600 > TIME_BUDGET_HOURS:
            raise TimeBudgetReached
    draw_dashboard()

## 8. Train

Same loop as the repo's `pairvc-train`. Every `eval_every` steps it validates,
writes an audio sample to `samples/` and saves `last.pt`, plus `best.pt` when
the validation mel loss improves.

**Interrupted, or hit the session limit?** Run the notebook again with
`RESUME = True`: it continues from `last.pt` (same step, same data order).
It also stops by itself after `TIME_BUDGET_HOURS`, right after a save.
On Kaggle, to continue in a *new* session, add the previous version's output
as input data and copy `checkpoints/run` back to `/kaggle/working` first.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)  # the dashboard replaces the log lines

last_path = OUTPUT_DIR / LAST_CHECKPOINT
if RESUME and last_path.exists():
    train_config = dataclasses.replace(train_config, resume=last_path)
    print("resuming from", last_path)

train_loader, val_loader = build_dataloaders(train_dataset, val_dataset, train_config, device)
start_time = time.perf_counter()
try:
    state = train(
        generator, discriminator, train_loader, val_loader,
        data_config, model_config, disc_config, train_config, callback=on_metrics,
    )
    draw_dashboard()
    print(f"Training finished. Best validation mel {state.best_val_mel:.4f}")
except TimeBudgetReached:
    draw_dashboard()
    print(f"Time budget of {TIME_BUDGET_HOURS} h reached; {last_path} is saved. Run the notebook again to continue.")

## 9. Results

### 9.1 Final curves
Saved as `training_curves.png` next to the checkpoints.

In [ ]:
curves_path = OUTPUT_DIR / "training_curves.png"
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r["step"] for r in history["train"]], [r["mel"] for r in history["train"]], label="train")
ax.plot([r["step"] for r in history["val"]], [r["mel"] for r in history["val"]], "o-", label="validation")
ax.set_title("mel loss vs teacher")
ax.set_xlabel("step")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(curves_path, dpi=120)
plt.show()
print("saved", curves_path)

### 9.2 Listen: convert validation clips with the best checkpoint
Loads `best.pt` exactly as the repo's `pairvc-convert` does, converts a few
validation sources to every target voice, and plays source, PairVC and teacher.

In [ ]:
best = torch.load(OUTPUT_DIR / BEST_CHECKPOINT, map_location="cpu", weights_only=True)
best_model = PairVC(ModelConfig(**best["model_config"]))
best_model.load_state_dict(best["generator"])
best_model.to(device).eval()
print(f"best.pt from step {best['step']:,} (validation mel {best['best_val_mel']:.4f})")

seen = set()
for pair in val_dataset.pairs:
    if pair.source in seen or len(seen) >= 3:
        continue
    seen.add(pair.source)
    source = load_audio(pair.source, data_config.sample_rate)
    print(f"\n=== {pair.source.name} ===\nsource:")
    display(Audio(source.numpy(), rate=data_config.sample_rate))
    for other in val_dataset.pairs:
        if other.source != pair.source:
            continue
        converted = best_model.convert(source[None].to(device), torch.tensor([other.speaker], device=device))[0]
        teacher = load_audio(other.target, data_config.sample_rate)
        print(f"voice {other.speaker}: PairVC, then teacher")
        display(Audio(converted.clamp(-1, 1).cpu().numpy(), rate=data_config.sample_rate),
                Audio(teacher.numpy(), rate=data_config.sample_rate))

### 9.3 Real-time check (CPU, 20 ms chunks)
Live conversion runs on a CPU, one 20 ms chunk at a time. **RTF** = compute
time / audio time: below 1 it keeps up with real time. (Colab/Kaggle CPUs are
slow and shared: a laptop is usually faster.)

In [ ]:
cpu_model = PairVC(best_model.config)
cpu_model.load_state_dict(best["generator"])
cpu_model.eval()
torch.set_num_threads(1)  # a live app usually gets one core
audio = load_audio(val_dataset.pairs[0].source, data_config.sample_rate)[: 5 * data_config.sample_rate]
chunk = model_config.hop_length  # 320 samples = 20 ms
states, spk, times = cpu_model.init_state(1), torch.tensor([0]), []
with torch.inference_mode():
    for start in range(0, len(audio) - chunk + 1, chunk):
        t0 = time.perf_counter()
        _, states = cpu_model.stream_step(audio[None, start:start + chunk], spk, states)
        times.append(time.perf_counter() - t0)
torch.set_num_threads(os.cpu_count() or 1)
rtf = sum(times) / (len(times) * chunk / data_config.sample_rate)
print(f"RTF {rtf:.3f} on 1 CPU thread | mean {1000 * sum(times) / len(times):.1f} ms, "
      f"worst {1000 * max(times):.1f} ms per 20 ms chunk | latency "
      f"{1000 * (chunk + model_config.lookahead_samples) / data_config.sample_rate:.0f} ms (chunk + lookahead)")

### 9.4 Objective metrics (same as `src/pairvc/metrics.py` and `src/pairvc/evaluate.py`)

Objective metrics: how close is PairVC's output to the teacher, and how natural is it?

All numbers compare a converted clip `x` with the teacher's clip `y` of the
same speech (they are time-aligned, so frames can be compared one to one):

- MCD (mel cepstral distortion, dB, lower is better): distance between the
  spectral envelopes, i.e. the *timbre*. Identical audio gives 0; values of
  ~4-6 dB are typical for good VC systems against their reference.
- F0 RMSE (cents, lower is better): pitch error on frames voiced in both clips.
  100 cents = 1 semitone; a model that does not change the pitch at all scores
  about 100 x the teacher's shift.
- Voicing agreement (%, higher is better): frames where both agree on "voiced"
  vs "unvoiced/silent".
- Mel L1: mean absolute log-mel difference (the training loss, at one scale).
- DNSMOS (optional, 1-5, higher is better): Microsoft's neural MOS predictor
  (P.835 SIG / BAK / OVRL), a stand-in for a listening test. Needs onnxruntime
  and downloads a 1.1 MB model once.

Everything here is plain numpy, no network (except DNSMOS), so it also runs in
the tests and on a laptop.

Score a trained model on a whole split: PairVC vs the teacher, with baselines.

    pairvc-eval --checkpoint checkpoints/run/best.pt --data-dir data --dnsmos

Every validation pair (whole clip, no cropping) is converted offline and
compared with the teacher's clip using pairvc.metrics. Three systems are
reported, so the numbers can be read without knowing the metrics:

- "no conversion": the source clip itself (what you hear without PairVC);
- "PairVC": the model;
- "teacher": the reference (0 distance by definition; its DNSMOS is the ceiling).

Pass `--untrained` to score a freshly initialised model with the same
architecture: the "before training" row of the README.

In [ ]:
import math
import urllib.request
from pathlib import Path
from typing import ClassVar

import numpy as np
from scipy.fft import dct
from scipy.signal import resample_poly


DNSMOS_URL = "https://github.com/microsoft/DNS-Challenge/raw/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx"


# ---------------------------------------------------------------------------
# Spectral features
# ---------------------------------------------------------------------------


def frames_of(audio: np.ndarray, frame: int, hop: int) -> np.ndarray:
    """[n_frames, frame] centred frames (zero-padded at both ends)."""
    audio = np.pad(np.asarray(audio, dtype=np.float64), (frame // 2, frame // 2))
    n = max(1, 1 + (len(audio) - frame) // hop)
    index = np.arange(frame)[None, :] + hop * np.arange(n)[:, None]
    return audio[index]


def log_mel(audio: np.ndarray, sample_rate: int, n_mels: int = 80) -> np.ndarray:
    """[n_frames, n_mels] natural-log mel spectrogram, 64 ms window, 16 ms hop."""
    n_fft = 2 ** round(math.log2(0.064 * sample_rate))
    spectrum = np.abs(np.fft.rfft(frames_of(audio, n_fft, n_fft // 4) * np.hanning(n_fft), axis=1))
    mel = spectrum @ mel_filterbank(sample_rate, n_fft, n_mels).numpy().T.astype(np.float64)
    return np.log(np.maximum(mel, 1e-5))


def energy_mask(audio: np.ndarray, sample_rate: int, threshold_db: float = -40.0) -> np.ndarray:
    """[n_frames] True where the frame is within `threshold_db` of the loudest one (speech)."""
    n_fft = 2 ** round(math.log2(0.064 * sample_rate))
    db = 10 * np.log10(np.mean(frames_of(audio, n_fft, n_fft // 4) ** 2, axis=1) + 1e-10)
    return db > db.max() + threshold_db


def mel_cepstrum(audio: np.ndarray, sample_rate: int) -> np.ndarray:
    """[n_frames, n_mels] real cepstrum of the log-mel spectrum: c_n = 1/N sum_k log S_k cos(...)."""
    log_spec = log_mel(audio, sample_rate)
    return dct(log_spec, type=2, axis=1) / (2 * log_spec.shape[1])  # scipy's DCT-II is 2 x the sum


def mel_cepstral_distortion(x: np.ndarray, y: np.ndarray, sample_rate: int, n_coeffs: int = 13) -> float:
    """MCD in dB between x and the reference y, on y's speech frames (c0, the loudness, excluded)."""
    n = min(len(x), len(y))
    cx, cy = mel_cepstrum(x[:n], sample_rate)[:, 1:n_coeffs], mel_cepstrum(y[:n], sample_rate)[:, 1:n_coeffs]
    mask = energy_mask(y[:n], sample_rate)
    per_frame = np.sqrt(2 * np.sum((cx - cy) ** 2, axis=1))
    return float(10 / math.log(10) * per_frame[mask].mean())


def mel_l1(x: np.ndarray, y: np.ndarray, sample_rate: int) -> float:
    """Mean |log10 mel(x) - log10 mel(y)|, the training loss at one scale."""
    n = min(len(x), len(y))
    return float(np.abs(log_mel(x[:n], sample_rate) - log_mel(y[:n], sample_rate)).mean() / math.log(10))


# ---------------------------------------------------------------------------
# Pitch
# ---------------------------------------------------------------------------


def estimate_f0(
    audio: np.ndarray, sample_rate: int, fmin: float = 60.0, fmax: float = 600.0, threshold: float = 0.2
) -> np.ndarray:
    """[n_frames] pitch in Hz (0 = unvoiced or silent), YIN algorithm, 16 ms hop.

    YIN: for each lag, how different is the frame from itself shifted by that lag
    (cumulative-mean normalised); the first dip below `threshold` is the period.
    """
    frame = 2 ** math.ceil(math.log2(2.5 * sample_rate / fmin))  # >= 2 periods of the lowest pitch
    hop = 2 ** round(math.log2(0.064 * sample_rate)) // 4  # same hop as log_mel
    frames = frames_of(audio, frame, hop)
    frames = frames - frames.mean(axis=1, keepdims=True)
    half = frame // 2
    # Difference function via FFT autocorrelation: d(tau) = r_0 + r_tau_energy - 2 r(tau).
    spectrum = np.fft.rfft(frames, n=2 * frame, axis=1)
    acf = np.fft.irfft(np.abs(spectrum) ** 2, axis=1)[:, : half + 1]
    energy = np.cumsum(frames[:, ::-1] ** 2, axis=1)[:, ::-1]  # energy of x[tau:]
    diff = acf[:, :1] + energy[:, : half + 1] - 2 * acf
    diff[:, 0] = 0
    cmnd = diff[:, 1:] * np.arange(1, half + 1) / np.maximum(np.cumsum(diff[:, 1:], axis=1), 1e-12)

    lag_min, lag_max = int(sample_rate / fmax), min(half - 1, int(sample_rate / fmin))
    f0 = np.zeros(len(frames))
    voiced_energy = energy_mask(audio, sample_rate)[: len(frames)]
    for i, row in enumerate(cmnd):
        if i >= len(voiced_energy) or not voiced_energy[i]:
            continue
        below = np.nonzero(row[lag_min:lag_max] < threshold)[0]
        if len(below) == 0:
            continue
        lag = lag_min + below[0]
        while lag + 1 < lag_max and row[lag + 1] < row[lag]:  # walk down to the bottom of the dip
            lag += 1
        f0[i] = sample_rate / (lag + 1)  # row index 0 is lag 1
    return f0


def f0_metrics(x: np.ndarray, y: np.ndarray, sample_rate: int) -> dict[str, float]:
    """Pitch error of x against the reference y: RMSE in cents and voicing agreement."""
    n = min(len(x), len(y))
    fx, fy = estimate_f0(x[:n], sample_rate), estimate_f0(y[:n], sample_rate)
    both = (fx > 0) & (fy > 0)
    cents = 1200 * np.log2(fx[both] / fy[both]) if both.any() else np.array([np.nan])
    return {
        "f0_rmse_cents": float(np.sqrt(np.mean(cents**2))),
        "voicing_agreement": float(np.mean((fx > 0) == (fy > 0))),
        "median_f0_hz": float(np.median(fx[fx > 0])) if (fx > 0).any() else 0.0,
    }


def compare(x: np.ndarray, y: np.ndarray, sample_rate: int) -> dict[str, float]:
    """Every reference metric of x against y in one dict."""
    return {
        "mcd_db": mel_cepstral_distortion(x, y, sample_rate),
        **f0_metrics(x, y, sample_rate),
        "mel_l1": mel_l1(x, y, sample_rate),
    }


# ---------------------------------------------------------------------------
# DNSMOS: predicted listening-test score (optional)
# ---------------------------------------------------------------------------


class DNSMOS:
    """Microsoft DNSMOS P.835 (non-intrusive: no reference needed). Scores on a 1-5 scale.

    SIG = speech quality, BAK = background noise, OVRL = overall. The model wants
    9 s at 16 kHz; shorter clips are repeated to fill 9 s, as the official script does.
    """

    SAMPLE_RATE = 16_000
    INPUT_SAMPLES = 144_160  # 9.01 s
    # Polynomials mapping the raw network outputs to MOS (from the official dnsmos_local.py).
    POLY: ClassVar[dict[str, tuple[float, float, float]]] = {
        "sig": (-0.08397278, 1.22083953, 0.0052439),
        "bak": (-0.13166888, 1.60915514, -0.39604546),
        "ovrl": (-0.06766283, 1.11546468, 0.04602535),
    }

    def __init__(self, model_path: Path | None = None) -> None:
        import onnxruntime as ort  # optional dependency: pip install 'tiny-pairvc[onnx]'

        if model_path is None:
            model_path = Path.home() / ".cache" / "pairvc" / "dnsmos_sig_bak_ovr.onnx"
        model_path = Path(model_path)
        if not model_path.exists():
            model_path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DNSMOS_URL, model_path)
        options = ort.SessionOptions()
        options.intra_op_num_threads = 1
        self.session = ort.InferenceSession(str(model_path), options, providers=["CPUExecutionProvider"])

    def __call__(self, audio: np.ndarray, sample_rate: int) -> dict[str, float]:
        if sample_rate != self.SAMPLE_RATE:
            g = math.gcd(sample_rate, self.SAMPLE_RATE)
            audio = resample_poly(audio, self.SAMPLE_RATE // g, sample_rate // g)
        audio = np.asarray(audio, dtype=np.float32)
        if len(audio) == 0 or not np.any(audio):
            return {"dnsmos_sig": 1.0, "dnsmos_bak": 1.0, "dnsmos_ovrl": 1.0}
        reps = math.ceil(self.INPUT_SAMPLES / len(audio))
        clip = np.tile(audio, reps)[: self.INPUT_SAMPLES][None]
        sig, bak, ovrl = self.session.run(None, {"input_1": clip})[0][0]
        scores = {}
        for name, raw in (("sig", sig), ("bak", bak), ("ovrl", ovrl)):
            a, b, c = self.POLY[name]
            scores[f"dnsmos_{name}"] = float(a * raw**2 + b * raw + c)
        return scores

In [ ]:
import argparse
import json
import logging
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import torch


logger = logging.getLogger(__name__)

SYSTEMS = ("no conversion", "PairVC", "teacher")


@torch.inference_mode()
def evaluate_split(
    model: PairVC,
    split_dir: Path,
    sample_rate: int,
    limit: int | None = None,
    dnsmos: DNSMOS | None = None,
    baselines: bool = True,
) -> dict[str, dict[str, float]]:
    """Mean metrics per system over the pairs of `split_dir`: {system: {metric: value}}.

    `baselines=False` scores only PairVC (the baselines do not depend on the model).
    """
    model.eval()
    device = next(model.parameters()).device
    pairs = find_pairs(Path(split_dir))[:limit]
    if not pairs:
        raise ValueError(f"no pairs in {split_dir}")
    systems = SYSTEMS if baselines else ("PairVC",)
    rows: dict[str, list[dict[str, float]]] = {name: [] for name in systems}
    for pair in pairs:
        source = load_audio(pair.source, sample_rate)
        target = load_audio(pair.target, sample_rate).numpy()
        speaker = torch.tensor([pair.speaker], device=device)
        converted = model.convert(source[None].to(device), speaker)[0].clamp(-1, 1).cpu().numpy()
        audios = dict(zip(SYSTEMS, (source.numpy(), converted, target), strict=True))
        for name in systems:
            audio = audios[name]
            scores = compare(audio, target, sample_rate)
            if dnsmos is not None:
                scores.update(dnsmos(audio, sample_rate))
            rows[name].append(scores)
    return {name: {k: float(np.nanmean([r[k] for r in rs])) for k in rs[0]} for name, rs in rows.items()}


def results_table(results: dict[str, dict[str, float]]) -> str:
    """A Markdown table, one row per system (the format used in the README)."""
    columns = [
        ("mcd_db", "MCD dB ↓", "{:.2f}"),
        ("f0_rmse_cents", "F0 RMSE cents ↓", "{:.0f}"),
        ("voicing_agreement", "voicing ↑", "{:.1%}"),
        ("mel_l1", "mel L1 ↓", "{:.3f}"),
        ("dnsmos_ovrl", "DNSMOS OVRL ↑", "{:.2f}"),
        ("dnsmos_sig", "DNSMOS SIG ↑", "{:.2f}"),
    ]
    columns = [c for c in columns if any(c[0] in r for r in results.values())]
    lines = [
        "| system | " + " | ".join(title for _, title, _ in columns) + " |",
        "|---|" + "---:|" * len(columns),
    ]
    for name, scores in results.items():
        cells = [
            "" if key not in scores else "–" if np.isnan(scores[key]) else fmt.format(scores[key])
            for key, _, fmt in columns
        ]  # "–": not measurable (e.g. no voiced frame, so no pitch)
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Score PairVC against the teacher on a data split.")
    p.add_argument("--checkpoint", type=Path, required=True, help="best.pt / last.pt / generator file")
    p.add_argument("--data-dir", type=Path, default=Path("data"))
    p.add_argument("--split", default="val")
    p.add_argument("--limit", type=int, default=None, help="only the first N pairs")
    p.add_argument("--untrained", action="store_true", help="same architecture, random weights")
    p.add_argument("--dnsmos", action="store_true", help="add DNSMOS (needs onnxruntime, downloads 1 MB)")
    p.add_argument("--output", type=Path, default=None, help="write the results as JSON here")
    p.add_argument("--device", default="cpu")
    p.add_argument("--seed", type=int, default=0, help="init seed for --untrained")
    return p


def run(argv: Sequence[str] | None = None) -> dict[str, dict[str, float]]:
    from pairvc.convert import load_model

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    args = build_parser().parse_args(argv)
    model, sample_rate = load_model(args.checkpoint, args.device)
    if args.untrained:
        torch.manual_seed(args.seed)
        model = PairVC(model.config).to(args.device)
    results = evaluate_split(
        model, args.data_dir / args.split, sample_rate, args.limit, DNSMOS() if args.dnsmos else None
    )
    print(results_table(results))
    if args.output is not None:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(json.dumps(results, indent=2))
        logger.info("wrote %s", args.output)
    return results


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point (returns None: console scripts pass the return value to sys.exit)."""
    run(argv)

In [ ]:
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])  # for DNSMOS
try:
    dnsmos = DNSMOS()
except Exception as error:  # no internet / no onnxruntime: the other metrics still work
    print("DNSMOS unavailable:", error)
    dnsmos = None

val_dir = Path(data_config.data_dir) / data_config.val_split
torch.manual_seed(train_config.seed)
untrained_model = PairVC(best_model.config).to(device)
before = evaluate_split(untrained_model, val_dir, data_config.sample_rate, limit=100, dnsmos=dnsmos)
after = evaluate_split(best_model, val_dir, data_config.sample_rate, limit=100, dnsmos=dnsmos, baselines=False)
results = {"no conversion": before["no conversion"], "PairVC, untrained": before["PairVC"],
           "PairVC, trained": after["PairVC"], "teacher": before["teacher"]}
(OUTPUT_DIR / "eval_results.json").write_text(json.dumps(results, indent=2))
display(Markdown(results_table(results)))

## 10. Download the trained model

Writes `pairvc_generator.pt` (only what inference needs, ~38 MB instead of
the full `best.pt` with discriminators and optimizer states) and zips it with
the metrics and curves. Use it on your computer with the repo:

```bash
unzip tiny-pairvc-run.zip -d checkpoints/run
pairvc-convert --checkpoint checkpoints/run/pairvc_generator.pt --input me.wav --speaker 0
pairvc-export  --checkpoint checkpoints/run/pairvc_generator.pt --output pairvc.onnx --verify
```

In [ ]:
import shutil
import tempfile

generator_path = OUTPUT_DIR / "pairvc_generator.pt"
torch.save({k: best[k] for k in ("generator", "model_config", "data_config", "step")}, generator_path)
with tempfile.TemporaryDirectory() as tmp:
    bundle = Path(tmp) / "bundle"
    bundle.mkdir()
    for name in ["pairvc_generator.pt", "metrics.json", "eval_results.json", "training_curves.png"]:
        if (OUTPUT_DIR / name).exists():
            shutil.copy(OUTPUT_DIR / name, bundle / name)
    archive = shutil.make_archive(str(OUTPUT_DIR.parent / "tiny-pairvc-run"), "zip", bundle)
print("created", archive)

if ON_COLAB:
    from google.colab import files

    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")